# <p style="background-color:#EFE4B0;font-family:Georgia;color:#990F02;font-size:150%;font-weight:bold;text-align:center;border-radius:10px 10px;">Прогноз кассовых сборов по данным из TMDB</p>
<center>
  <div style="display: flex; justify-content: center; align-items: center; gap: 20px;">
    <img src="https://www.nikproject.com/images/top-10/kasfilmall/b-kasfilmall.jpg"
         width="1000" 
         height="400">
    <img src="https://kodi.tv/images/addons/omega/context.embuary.info/resources/icon.png"
         width="500" 
         height="800">
  </div>
</center>


### <p style="margin-top:20px; background-color:#EFE4B0; background-size:50%; font-weight:bold; font-family:Georgia;color:#990F02;font-size:120%;border-radius:5px 5px;display:inline-block">Описание набора данных</p>

<p style="font-family:Georgia;color:#990F02;font-size:120%; margin-top: 10px">Набор данных в этой задаче собран по аналогии данных для конкурса на платформе<a href="https://www.themoviedb.org/" style="color:#228B22;"> Kaggle - The Movie database</a>.
    <br>Данные представляют более чем 20000 фильмов, выпущенных с 1991 по 2025 гг и разнообразные метаданные, полученные из <a href="https://www.themoviedb.org/documentation/api" style="color:#228B22;">The Movie database (TMDB)</a>. Фильмы имеют идентификаторы (id). Включенные данные содержат информацию о составе актеров, съемочной группе, ключевых словах сюжета, бюджете, постерах, датах выхода, языках, производственных компаниях и странах. Нам предстоит построить модель, которая будет предсказывать мировую кассу для предстоящих фильмов.</p>

## Loading libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from time import ctime
import requests
import json
from dotenv import load_dotenv
import ast
from typing import *
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import plotly.express as px
import plotly.graph_objs as go
import plotly.offline as py
py.init_notebook_mode(connected=True)

from PIL import Image
from urllib.request import urlopen
from matplotlib.colors import ListedColormap
# Стилистика и дизайн ноутбука
fontdict={"fontfamily": "arial","color": "#682F2F", "fontsize": 20}

import seaborn as sns
from pprint import pprint
from collections import Counter 
from wordcloud import WordCloud
from time import sleep


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error as mae, mean_absolute_percentage_error as mape, root_mean_squared_error as rmse, r2_score 
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna


import warnings
warnings.filterwarnings("ignore")

## Configs

In [ ]:
load_dotenv("configs.env")
API_KEY = os.getenv("API_KEY")

fontdict={"fontfamily": "arial","color": "#682F2F"}

cmap = ListedColormap([
    "#682F2F", "#9E726F", "#D6B2B1", "#B9C0C9", "#9F8A78", "#F3AB60", "#EFE4B0",
    "#6A7BA2", "#4B3F37", "#F7F4ED", "#CFA27E", "#E2D7C5", "#808000",  "#B2D3A8","#3F5E64"
])

palette = [
    "#682F2F", "#9E726F", "#D6B2B1", "#B9C0C9", "#9F8A78", "#F3AB60", "#EFE4B0",
    "#6A7BA2", "#4B3F37", "#F7F4ED", "#CFA27E", "#E2D7C5", "#808000",  "#B2D3A8","#3F5E64"]

display(cmap)

## Load data

In [ ]:
raw_data = pd.read_csv("data/tmdb_movies_1990_2025.zip", index_col=0)
raw_data.shape

In [ ]:
# Скопируем данные, чтобы "под рукой" всегда были сырые данные
data = raw_data.copy()
data.head()

Проверим, не появились ли дубликаты фильмов при сборе данных. Это возможно, поскольку в исходной базе один и тот же фильм может встречаться несколько раз из-за изменения показателей, например,"popularity".

In [ ]:
pd.options.display.max_rows = 100
data[data.duplicated(subset='id', keep=False)].sort_values(by='id')[['id', 'title', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'release_date']]

Вижу пока только одну причину образования дубликатов по id фильма, это изменение рейтинга "popularity", однако у некоторых фильмов этот признак идентичен в дубликатах, возможно есть другие причины.

Оставим только последние вхождения дубликатов, так как они должны быть актуальнее всех.

In [ ]:
data = data.drop_duplicates(subset='id', keep='last').reset_index(drop=True)
data.shape

Если разделить данные по аналогии с Kaggle, где тестовая выборка содержит фильмы, выпущенные на год позже обучающих, то получится следующее:
фильмы с 1990 по 2024 год войдут в обучающую выборку, а фильмы с 1991 по 2025 год — в тестовую.
Таким образом, картины 1990 года полностью попадут в обучающую выборку, а 2025 года — в тестовую.
Остаётся случайным образом разделить фильмы, выпущенные в период с 1991 по 2024 годы, в пропорции 60% для обучения и 40% для теста. 

Для деления данных по дате выпуска, сначла поправим признак. Начнем с пропуков.

## Check feature release date

### Missing data in release_date

In [ ]:
data[data['release_date'].isna()][['id', 'budget', 'revenue', 'title', 'release_date']]

Признак даты выпуска фильмов имеет один пропуск, дату выпуска одного фильма, попробую посмотреть на самой базе TMDB, только через другой эндпоинт - "https://api.themoviedb.org/3/movie"

In [ ]:
url = "https://api.themoviedb.org/3/movie/{}"
id = 1564897


params = {
    "api_key": API_KEY,
    "language": "en-US"
}
response = requests.get(url.format(id), params=params)
if response.status_code==200:
    result = response.json()
else:
    print(f"Ошибка: {response.status_code}, {response.text}")
release_date = result.get("release_date")
print(f"missing_release_date: {release_date}")

In [ ]:
data['release_date'] = data['release_date'].fillna(release_date, axis=0)
data['release_date'].isna().sum()

Прежде чем разделить данные по дате релиза, проверим этот признак

In [ ]:
print(sorted(data['release_date'].apply(lambda x: str(x).split("-")[0]).unique()))
print()
print(sorted(data['release_date'].apply(lambda x: str(x).split("-")[1]).unique()))
print()
print(sorted(data['release_date'].apply(lambda x: str(x).split("-")[2]).unique()))

Первое число обозначает год, второе — месяц, а третье — день. Аномальных значений нет, формат данных не смешан , по этому можем смело работать с данными как с типом данных datetime.

In [ ]:
data['release_date'] = pd.to_datetime(data['release_date'], format="%Y-%m-%d")
data = data.sort_values(by='release_date', ascending=True).reset_index(drop=True)

## Split the data into training and test sets

In [ ]:
start_filter = data['release_date']>=pd.to_datetime("1991-01-01", format="%Y-%m-%d")
end_filter = data['release_date']<=pd.to_datetime("2024-12-31", format="%Y-%m-%d")
test_sample = data[start_filter&end_filter]


rest_train_data = data[data['release_date']<pd.to_datetime("1991-01-01", format="%Y-%m-%d")]
rest_test_data = data[data['release_date']>pd.to_datetime("2024-12-31", format="%Y-%m-%d")]



print(f"Количество фильмов выпущенных с 1991 по 2024 годах: {test_sample.shape[0]}")
print(f"Количество фильмов выпущенных раньше 1991 года в датасете: {rest_train_data.shape[0]}")
print(f"Количество фильмов выпущенных в 2025 году в датасете: {rest_test_data.shape[0]}")

Так как 40% общей выборки составляют около 8000 ($(18930*40)/100 = 7 572$) фильмов, случайным образом выбираем их и добавляем к ним «хвост» — фильмы последних лет выпуска.

In [ ]:
random_indexes = np.random.choice(test_sample.index, 8000, replace=False)
subset = test_sample.loc[random_indexes]
train_subset = test_sample.drop(index=subset.index)



# Объединяем тестовую выборку из общего диапазона и фильмы 2025 года
test = pd.concat([subset, rest_test_data], axis=0).reset_index(drop=True)
print(f"Размерность тестовой выборки: {test.shape}")

# Обединяем остальные картины из общей выборки и "начало" данных
train = pd.concat([rest_train_data, train_subset], axis=0).reset_index(drop=True)
print(f"Размерность обучающей выборки:{train.shape}")

## Overview

In [ ]:
(train.isna().sum()/train.shape[0]*100).__round__(2)

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train[['budget', 'revenue']].describe().T

In [ ]:
test[['budget', 'revenue']].describe().T

В обчающей выборке есть фильмы с отрицательным целевым признаком - 'revenue'. Посмотрим на них.

In [ ]:
train[train['revenue']<0][['id', 'budget', 'revenue', 'title', 'release_date']]

Таких фильмов всего один, удалим его.

In [ ]:
train = train.drop(index=train[train['revenue']<0].index)

### Feature **release_date** 

Извлечем из признака отдельные признаки года, месяца, недели, квартала и дня

In [ ]:
# Функция для извлечения всех временных признаков
def date_decomposition(df):
    time_components = ['year', 'month', 'weekday', 'day', 'quarter']
    
    # weekofyear отдельно обработаем
    for component in time_components:
        name_component = 'release_date_' + component
        df[name_component] = getattr(df['release_date'].dt, component).astype(int)
    
    # неделя года (isocalendar)
    df['release_date_weekofyear'] = df['release_date'].dt.isocalendar().week.astype(int)
    
    return df


train = date_decomposition(train)
test = date_decomposition(test)
 
# Составляем список столбцов для удаления избавляемся сразу от признака даты первоначального выпуска, 
# т.к. нам достаточно одной даты выпуска
columns_to_drop = ['release_date', 'primary_release_date']

In [ ]:
train.columns

In [ ]:
# Average revenue by month

fig = plt.figure(figsize=(10,10))

train.groupby('release_date_month')['revenue'].agg('mean').plot(kind='bar',color=cmap.colors[13:14],rot=0)
plt.ylabel('Revenue (100 million dollars)')
plt.title("Средняя выручка и месяц выпуска фильмов", fontdict=fontdict)
plt.show();

Похоже, что наибольшие кассовые сборы приходятся на июль, декабрь, июнь и май. Возможно, это связано с тем, что именно в эти месяцы зрители чаще готовы тратить деньги и время на походы в кино — ведь они совпадают с отпускным и праздничным периодами.

### All features that contain a list of dictionaries 

Дата-фрейм имеет признаки - списки с вложенными словарями, изучим их следующим этапом. 

Так как они хранятся в виде строковых значений, с ними работать как со списком или словарем невозможно, по этому переведем их в списки, а пропуски в пустые словари.

In [ ]:
%%time
dict_features = ["belongs_to_collection", 
                 "genres", 
                 "production_companies", 
                 "production_countries", 
                 "spoken_languages", 
                 "keywords",
                 "cast",
                 "crew"
                ]

def string_to_dict(df):
    for feature in dict_features:
        df[feature] = df[feature].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else {})
    return df

train = string_to_dict(train)
test = string_to_dict(test)

### Feature - **belongs_to_collection** 

In [ ]:
for i, j in enumerate(train['belongs_to_collection'][6:15]):
    print(i, j)

Это поле можно преобразовать в бинарный признак, указывающий является ли фильм коллекционным, где 1 - Да, 0 - Нет, а ткже можно извлечь название коллекции, при этом остальные объекты (ID, poster_path, backdrop_path) в поле не несут общей информативной нагрузки.

In [ ]:
# Бинарный признак принадлежности к коллекции
train['has_collection'] = train['belongs_to_collection'].apply(lambda x: 1 if x!={}  else 0)
test['has_collection'] = test['belongs_to_collection'].apply(lambda x: 1 if x!={}  else 0)


# Признак с наименованием коллекции
train['collection_name'] = train['belongs_to_collection'].apply(lambda x: x['name'] if x!={} else "hasn't a collection")
test['collection_name'] = test['belongs_to_collection'].apply(lambda x: x['name'] if x!={} else "hasn't a collection")


# Составляем список удаляемых столбцов
columns_to_drop.append('belongs_to_collection')
columns_to_drop

### Feature - **genres**

In [ ]:
for i, j in enumerate(train['genres'][:10]):
    print(i, j)

In [ ]:
text = ' '.join(i['name'] for g in train['genres'] if g!=[] for i in g)

wordcloud = WordCloud(colormap=cmap, background_color='white', collocations=False,
                      width=1600, height=1200).generate(text)

plt.figure(figsize=(15,6))
plt.imshow(wordcloud)
plt.title('Top genres')
plt.axis("off")
plt.show();

Наиболее популярные жанры это - Drama, Comedy и Thriller

In [ ]:
s = [i['name'] for g in train['genres'] if g!=[] for i in g]
freq_genres = Counter(s).most_common()
print("Количество уникальных жанров", len(freq_genres))
freq_genres

Для отбора топ-жанров в качестве отдельных признаков можно проанализировать их кумулятивный вклад в общий процент и оставить только те жанры, которые обеспечивают наибольший рост этого показателя.

In [ ]:
percent_genres = []
n = len(s)
for i, j in freq_genres:
    percent_genres.append((i, round(j/n*100, 2)))

plt.figure(figsize=(15,5))
names = [name for name, percent in percent_genres] # наименования всех жанров
cumsum_percents = np.cumsum([percent for name, percent in percent_genres]) # кумумлятивная сумма процентов
sns.lineplot(x=names, y=cumsum_percents, color="#682F2F")
plt.xticks(names, rotation=45, color="#682F2F")
plt.title("Кумулятивный вклад каждого жанра в общий процент", fontdict=fontdict)
plt.show()

Можно сказать, что график выравнивается после жанра "war". 
Возьмём все жанры, исключая последние 3 - начиная с war. 

In [ ]:
top_genres = names[:-3]
print("Количество добавляемых фичей по жанрам", len(top_genres))
top_genres

In [ ]:
# Создадим функцию для формирования новых бинарных фичей с горячей кодировкой
def get_binary_features(top_features: List[str], df: pd.DataFrame, lists_of_features: List[list], prefix: str) -> pd.DataFrame:
    for feature in top_features:
        feature_name = prefix + feature
        df[feature_name] = lists_of_features.apply(lambda x: 1 if feature in x else 0)
    return df

In [ ]:
# Формируем серии, где каждое значение — список жанров, к которым относится соответствующий фильм.
train_lists_of_genres = train['genres'].apply(lambda x: [i['name'] for i in x] if x!=[] else 'no genres')
test_lists_of_genres = test['genres'].apply(lambda x: [i['name'] for i in x] if x!=[] else 'no genres')

# Формируем новые признаки для модели
train = get_binary_features(top_genres, train, train_lists_of_genres, 'genre_')
test = get_binary_features(top_genres, test, test_lists_of_genres, 'genre_')

# Добавляем признак, содержащий число жанров, к которым относится фильм.
train['NumOfGenres'] = train_lists_of_genres.apply(lambda x: len(x) if x!=[] else 0)
test['NumOfGenres'] = test_lists_of_genres.apply(lambda x: len(x) if x!=[] else 0)


# Добавим также признак, содержащий все жанры фильма
train['all_genres'] = train['genres'].apply(lambda x: ' '.join(sorted([i['name'] for i in x])) if x != [] else 'no genres')
test['all_genres'] = test['genres'].apply(lambda x: ' '.join(sorted([i['name'] for i in x])) if x != [] else 'no genres')

# Список удалемых столбцов
columns_to_drop.append("genres")
print("Список удаляемых столбцов",  columns_to_drop)

train.columns

### Fature - **homepage**

In [ ]:
train['homepage']

Данное поле содержит уникальные значения для каждого фильма - ссылку на официальный сайт, так что оно не несет информативной нагрузки для модели, однако можно классифицировать фильмы на фильмы с ссылкой и без ссылки.

In [ ]:
train['has_homepage'] = train['homepage'].apply(lambda x: 1 if not pd.isna(x) else 0)
test['has_homepage'] = test['homepage'].apply(lambda x: 1 if not pd.isna(x) else 0)


# Пополняем список удалемых столбцов
columns_to_drop.append("homepage")
print("Список удаляемых столбцов",  columns_to_drop)

Проверим связь между выручкой и наличием страницы фильма.

In [ ]:
sns.catplot(data=train, y='revenue', x="has_homepage", hue='has_homepage', palette=cmap.colors[13:])
plt.title("Выручка в зависимости от наличия домашней страницы", fontdict=fontdict)
plt.show()

Наблюдается слабая положительная зависимость между наличием домашней страницы и выручкой фильма — проекты с официальным сайтом, как правило, имеют более высокие кассовые сборы. Однако эта зависимость может быть обусловлена тем, что фильмов с домашней страницей в целом больше, и высокая выручка может объясняться именно большей долей таких фильмов в выборке.

### Feature - **poster_path** and **imdb_id**

In [ ]:
train['poster_path']

Подставив этот путь к базовому адресу "https://image.tmdb.org/t/p/w500", можно получить постеры — изображения, представляющие фильмы. Таким образом, можно визуализировать топ самых кассовых фильмов.

In [ ]:
# Посмотрим на топ 16 фильмов
top_revenue_films = data[['id', 'title', 'poster_path', 'revenue']]\
                        .sort_values(by='revenue', ascending=False)\
                        .head(16)\
                        .reset_index(drop=True)

fig = plt.figure(figsize=(48, 40))
for i, film_info in enumerate(top_revenue_films.itertuples(index=False)):
    id, name, path, revenue = film_info
    if not path:
        continue
    try:
        im = Image.open(urlopen(f"https://image.tmdb.org/t/p/w500{path}"))
    except Exception as e:
        print(f"Ошибка {name}, id - фильма {id} {e}")
        continue
    ax=fig.add_subplot(4,4, i+1, xticks=[], yticks=[])
    ax.imshow(im)
    ax.set_title(name, fontdict={
                                "fontsize": 40,
                                "fontname": "Arial",
                                "fontweight": "bold",  
                                "color": "#990F02"    
                            })

        
plt.tight_layout()
plt.show();      

In [ ]:
columns_to_drop.extend(['poster_path', 'imdb_id'])
columns_to_drop

### Feature - **production_companies**

In [ ]:
train['production_companies']

In [ ]:
s = [i['name'] for c in train['production_companies'] if c !=[] for i in c]
freq_companies = Counter(s).most_common()
len(freq_companies)

In [ ]:
percents = []
companies = []
n = len(s)
for company, freq in freq_companies:
    percents.append(round(freq/n*100, 2))
    companies.append(company)

cumsum_percents = np.cumsum(percents)
plt.figure(figsize=(15,6))
sns.lineplot(x=companies, y=cumsum_percents, color="#682F2F")
plt.xticks([])
plt.title("Сumulative contribution of each company", fontdict=fontdict)
plt.axhline(y=12, color='red', linewidth=1.5, linestyle='--')
plt.show();

In [ ]:
np.sum(cumsum_percents <=12)

Так как компаний очень много, возьмём только топ-40 кинокомпаний, чьи кумулятивные вклады обеспечили вертикальный рост. Можно было взять и выше, но объём выборки у нас небольшой, по этому во избежание "Проклятия размерности" не будем сильно раздувать простраство фичей.

In [ ]:
top_companies = companies[:40]
top_companies

In [ ]:
# Формируем серии, где каждое значение — список компаний - основателей каждого фильма.
train_lists_of_companies = train['production_companies'].apply(lambda x: [i['name'] for i in x ] if x!=[] else ['no production_companies'])
test_lists_of_companies = train['production_companies'].apply(lambda x: [i['name'] for i in x ] if x!={} else ['no production_companies'])


# Создаём отдельные фичи горячей кодировкой
train = get_binary_features(top_companies, train, train_lists_of_companies, "prod_company ")
test = get_binary_features(top_companies, test, test_lists_of_companies, "prod_company ")


# Добавим признаки по количеству компаний по каждому фильму
train['NumOfCompanies'] = train_lists_of_companies.apply(lambda x: len(x))
test['NumOfCompanies'] = test_lists_of_companies.apply(lambda x: len(x))


# Дополним список фичей для удаления
columns_to_drop.append("production_companies")
columns_to_drop

### Feature - **production countries**

In [ ]:
pd.set_option('display.max_colwidth', None)
train['production_countries'][194:210] # для примера берем рандомный срез, чтобы посмотреть как выглядят  пустые списки

In [ ]:
s = [i['name']  for countries in train['production_countries'] if countries != [] for i in countries]
freq_countries = Counter(s).most_common()
print("Количество стран, выпускающих фильмы", len(freq_countries))

In [ ]:
n=len(s)
countries = []
percents = []
for country, freq in freq_countries:
    countries.append(country)
    percents.append(round(freq/n*100, 2))

cumsum_percents = np.cumsum(percents)
plt.figure(figsize=(15,6))
sns.lineplot(x=countries, y=cumsum_percents, color="#682F2F")
plt.xticks([])# countries, rotation=90
plt.title("Сumulative contribution of each countries")
plt.axhline(y=85, color='red', linestyle="--", linewidth=1.5)
plt.show();

In [ ]:
np.sum(cumsum_percents <= 85)

Выбрав топ-25 стран по их вкладу в рост дохода, можно охватить основную часть  роста выручки.

In [ ]:
top_countries = countries[:25]
top_countries

In [ ]:
# Формируем серии, где каждое значение — список стран каждого фильма.
train_lists_of_countries = train['production_countries'].apply(lambda x: [i['name'] for i in x] if x!=[] else ['no countries'])
test_lists_of_countries = test['production_countries'].apply(lambda x: [i['name'] for i in x] if x!=[] else ['no countries']) 

# Создаём отдельные фичи горячей кодировкой
train = get_binary_features(top_countries, train, train_lists_of_countries, "prod_country ")
test = get_binary_features(top_countries, test, test_lists_of_countries, "prod_country ")

# Добавим признаки по количеству стран по каждому фильму
train['NumOfCountries'] = train_lists_of_countries.apply(lambda x: len(x))
test['NumOfCountries'] = test_lists_of_countries.apply(lambda x: len(x))


# Дополним список фичей для удаления
columns_to_drop.append("production_countries")
columns_to_drop

### Feature - **spoken_languages**

In [ ]:
train['spoken_languages'].head(10)# проверка пустых списков в данных

In [ ]:
train_lists_of_languages = train['spoken_languages'].apply(lambda x: [i['english_name'] for i in x] if x!=[] else ['no languages'])
test_lists_of_languages = test['spoken_languages'].apply(lambda x: [i['english_name'] for i in x] if x!=[] else ['no languages'])

train_lists_of_languages

In [ ]:
mask = train_lists_of_languages.apply(lambda x: x == ['no languages'])
train_lists_of_languages[mask].shape

In [ ]:
freq_langs = Counter([i for j in train_lists_of_languages if j !=['no languages'] for i in j]).most_common() 
print(len(freq_langs))
freq_langs[:25]

In [ ]:
# получим сразу фичи по количеству языков в фильме
train['NumOfLanguages'] = train_lists_of_languages.apply(lambda x: len(x))
test['NumOfLanguages'] = test_lists_of_languages.apply(lambda x: len(x))



# Построим для наглядности кумулятивный вклад каждого языка в рост выручки
n = train['NumOfLanguages'].sum()
langs = [i for i, j in freq_langs[:50]]
percents = [round(j/n*100, 2) for i, j in freq_langs[:50]]
cumsum_percents = np.cumsum(percents)


plt.figure(figsize=(15,6))

sns.lineplot(x=langs, y = cumsum_percents, color="#682F2F")
plt.title("Сumulative contribution of each languages")
plt.xticks(langs, rotation=90)



# Для отображения иероглифов на графике
font_path = "fonts/NotoSansJP-VariableFont_wght.ttf"
font_path_2 = "fonts/NotoSansKR-VariableFont_wght.ttf" 
font_path_3 = "fonts/NotoSansTC-VariableFont_wght.ttf"

fm.fontManager.addfont(font_path)
fm.fontManager.addfont(font_path_2)
fm.fontManager.addfont(font_path_3)

# Название шрифта 
prop = fm.FontProperties(fname=font_path)
prop_2 = fm.FontProperties(fname=font_path_2)
prop_3 = fm.FontProperties(fname=font_path_3)


plt.rcParams['font.family'] = prop.get_name()
plt.rcParams['font.family'] = prop_2.get_name()
plt.rcParams['font.family'] = prop_3.get_name()


plt.axhline(y=90, color='red', linewidth=1.5, linestyle="--")
plt.show();

In [ ]:
np.sum(cumsum_percents <= 90)

Думаю достаточно взять 25 топ  языков в качестве отдельных фичей.

In [ ]:
top_langs = [i[0] for i in freq_langs[:25]]
top_langs

In [ ]:
# Создаём отдельные фичи горячей кодировкой
train = get_binary_features(top_langs, train, train_lists_of_languages, "spoken_langs_")
test = get_binary_features(top_langs, test, test_lists_of_languages, "spoken_langs_")


# Дополним список фичей для удаления
columns_to_drop.append("spoken_languages")
columns_to_drop

### Feature - **keywords**

In [ ]:
train['keywords'].head()

In [ ]:
s = [i['name'] for k in train['keywords'] if k!=[] for i in k]
print("количество уникальных ключевых слов", len(np.unique(s)))

In [ ]:
text = ' '.join(['_'.join(i.split(' ')) for i in s])
wordcloud = WordCloud(colormap=cmap, background_color='white', collocations=False,
                      width=1600, height=1200).generate(text)
plt.figure(figsize=(12, 8))
plt.imshow(wordcloud)
plt.title('Top keywords')
plt.axis("off")
plt.show();

Можно заметить, что среди наиболее часто встречающихся ключевых слов лидируют такие, как «на основе романа или книги», «женщина-директор», «продолжение», «основано на реальных событиях», «смерть», «месть», а также темы, связанные с взаимоотношениями между членами семьи и другими подобными сюжетными линиями.

В качестве отдельных фичей можем взять количество ключевых слов и топ 30 слов, опять таки во избежания сильного роста числа признаков.

In [ ]:
freq_keywords = Counter(s).most_common()
freq_keywords[:30]

In [ ]:
# Формируем серии, где каждое значение — список ключевых слов каждого фильма.
train_lists_of_keywords = train['keywords'].apply(lambda x: [i['name'] for i in x] if x!=[] else ['no keywords'])
test_lists_of_keywords = test['keywords'].apply(lambda x: [i['name'] for i in x] if x!=[] else ['no keywords'])

# Создаём отдельные фичи горячей кодировкой
top_keywords = [i[0] for i in freq_keywords[:30]]
train = get_binary_features(top_keywords, train, train_lists_of_keywords, "keyword ")
test = get_binary_features(top_keywords, test, test_lists_of_keywords, "keyword ")



# Добавим признаки по количеству стран по каждому фильму
train['NumOfKeywords'] = train_lists_of_keywords.apply(lambda x: len(x))
test['NumOfKeywords'] = test_lists_of_keywords.apply(lambda x: len(x))

# Дополним список фичей для удаления
columns_to_drop.append("keywords")
columns_to_drop

### Feature - **cast**

In [ ]:
train['cast'].head()

**cast** — это состав актёров фильма. Эти данные представляют информацию об участии конкретного актёра в этом фильме, имя героя, а также персональные данные, такие как: имя, пол, профильная фотография (poster) и информация о популярности актёра. 

Нельзя однозначно утверждать, что популярность или мастерство актёра определяют уровень доходности фильма — скорее, этот показатель в большей степени зависит от объёма вложенного бюджета. Поэтому для начала рассмотрим, как соотносятся бюджет и выручка в данных. 

In [ ]:
fig = px.scatter(
    train,
    x="revenue",
    y="budget",
    color_discrete_sequence=["#9E726F"],  # цвет точек
    title="Распределение выручки в зависимости от бюджета"
)

fig.update_layout(
    title_font=dict(family="Arial", size=20, color="#682F2F"),
    width=1000,
    height=600
)

fig.show();

Наблюдается ожидаемая закономерность — с ростом бюджета увеличивается и выручка фильма. Проанализируем, фильмы с какими актёрами оказались самыми затратными и самыми прибыльными.

#### Celebrity income ranking

Используя атрибут poster_path и основной эндпоинт для изображений, можно получить профильные фотографии актёров. Таким образом можно визуально увидеть топ-знаменитостей (по выручке) этого периода. 

Отберем актёров из топ кассовых фильмов

In [ ]:
top_films = train[train['revenue']>=1000000000].sort_values(by="revenue", ascending=False).reset_index(drop=True)
print(f"Всего фильмов с выручкой более $1 млрд: {top_films.shape[0]}")
top_films[['title','revenue']].head(15)

In [ ]:
list_of_top_cast = top_films['cast'].apply(
    lambda x: [(i.get("name"), i.get("profile_path"), i.get("id")) for i in x if i.get("order")==0] if x!={} and x else []
)


# Подсчёт наиболее часто встречающихся актёров
freq_actors_celebrities = Counter([i for j in list_of_top_cast for i in j]).most_common(32)


fig = plt.figure(figsize=(54, 48))
for i, (actor_info, _) in enumerate(freq_actors_celebrities):
    name, path, id = actor_info
    if not path:
        continue
    try:
        ax = fig.add_subplot(6, 6, i+1, xticks=[], yticks=[])  
        im = Image.open(urlopen(f"https://image.tmdb.org/t/p/w500{path}"))
        ax.imshow(im)
        ax.set_title(name, fontdict={
                                    "fontsize": 40,
                                    "fontname": "Arial",
                                    "fontweight": "bold",  
                                    "color": "#990F02"    
                                })
    except Exception as e:
        print(f"Ошибка {name}, id-{id}: {e}")
plt.tight_layout()
plt.show();

Однако основное внимание следует уделить не единичным выбросам, а основной массе фильмов.
Посмотрим на топ-актёров  по количеству участий в фильмах, а не по топ-выручке.

#### Top celebrities by number of films

In [ ]:
# Список актёров в каждом фильме
list_of_cast = train['cast'].apply(
    lambda x: [(i.get('name'), i.get('profile_path'), i.get('id')) for i in x] if x!={}  and x else []
)

# Подсчёт уникальных актёров и топ 20 из них
freq_actors_overall = Counter([i for j in list_of_cast for i in j]).most_common(20)


# Визуализация постеров
fig = plt.figure(figsize=(40, 32))  
for i, actor_info in enumerate(freq_actors_overall):
    info, _ = actor_info
    name, path, ids = info
    if not path:
        continue
    try:
        ax = fig.add_subplot(5, 4, i+1, xticks=[], yticks=[])  
        im = Image.open(urlopen(f"https://image.tmdb.org/t/p/w500{path}"))
        ax.imshow(im)
        ax.set_title(name, fontdict={
            "fontsize": 40,
            "fontname": "Arial",
            "fontweight": "bold",  
            "color": "#990F02"    
        })
    except Exception as e:
        print(f"Олибка при запросе постера {i["name"]}; {e}")
        continue
    sleep=(0.2)
plt.tight_layout()
plt.show()

#### gender distribution

In [ ]:
list_of_gender = train['cast'].apply(lambda x:[i['gender'] for i in x if x!=[]])
s = [gender for i in list_of_gender for gender in i]
Counter(s).most_common()

* 0: "unknown" - это неуточненные данные, т.к. база TMBD заполняется пользователями, то некоторые атрибуты могут быть не внесены.
(https://www.kaggle.com/c/tmdb-box-office-prediction/discussion/80983#475572), 

* 1: "female" - женский пол,

* 2: "male" - мужской пол,

* 3: "non-binary" -  означает, что человек не идентифицирует себя исключительно как мужской или женский пол.


Полагаю, третий тип гендерного разделения мог бы проявить зависимость с целевым признаком, если бы его представление в данных было более значительным, но их не так много. Это связано с тем, что тема толерантности к подобным различиям остаётся острой, и мнения зрителей по этому поводу существенно расходятся.  

Изучим статистику таких фильмов, и число актеров с этим идентификатором.

In [ ]:
# Список актёров относящих себя к гендеру Non-binary
gender_3 = [(i['name'], i['profile_path']) for j in train['cast']  for i in j if i['gender']==3]
# Подсчёт количества уникальных актёров с гендерным идентификатором Non-binary
unique_gender_3 = Counter(i for i in gender_3).most_common()
print(f"Количество актёров с гендерным идентификатором - Non-binary в целом в базе TMDB:{len(unique_gender_3)}")


# Список актёров в главной роли с гендерным идентификатором Non-binary
gender_3_order_0 = [(i['name'], i['profile_path']) for j in train['cast']  for i in j if (i['gender']==3)and(i['order']==0)]
print(f"Фильмов, в которых в главной роли играет актёр с гендерным признаком None binary: {len(gender_3_order_0)}")
# Количество уникальных актёров в главной роли с гендерным идентификатором Non-binary
unique_gender_3_order_0 = Counter(i for i in gender_3_order_0).most_common()
print(f"Количество актёров с гендерным идентификатором - Non-binary  в главных ролях:{len(unique_gender_3_order_0)}")

fig = plt.figure(figsize=(48, 40))  
for i, actor_info in enumerate(unique_gender_3_order_0):
    info, freq = actor_info
    name, path = info
    if not path:
        continue
    try:
        ax = fig.add_subplot(6, 5, i+1, xticks=[], yticks=[])  
        im = Image.open(urlopen(f"https://image.tmdb.org/t/p/w500{path}"))
        ax.imshow(im)
        ax.set_title(name, fontdict={
            "fontsize": 40,
            "fontname": "Arial",
            "fontweight": "bold",  
            "color": "#990F02"    
        })
    except Exception as e:
        print(f"Олибка при запросе постера {i["name"]}; {e}")
        continue
    sleep=(0.2)
plt.tight_layout()
plt.show();

Записей gender = 3 встречается во всей выборке меньше 150, думаю, это небольшое число, и их можно объединить с записями gender = 0 в категорию “unknown/non-binary”.

Из данных об актёрском составе можно извлечь следующие признаки:

1. Количество актёров, задействованных в фильме.
2. Наличие в актёрском составе звёзд из списка топ-знаменитостей (их отобрано 20), а также идентификация конкретных имён.
3. Распределение актёров по полу — доля мужчин, женщин и неизвестных в составе.
4. Признаки по ролям актёров “character” – это не всегда имя, чаще это должность, псевдоним, статус: например, доктор, медсестра, он, она, официантка, бармен, танцор… т.д. 

Остальные данные, содержащиеся в этом наборе, предположительно не обладают высокой информативной ценностью для построения модели.

####  Feature engineering

Объединив список знаменитостей с наибольшей кассовой выручкой и список тех, кто чаще всего участвует в фильмах, сформируем итоговый список топ-актёров.

In [ ]:
celebr_actors = [i[0][0] for i in freq_actors_celebrities ] # топ знаменитых актёров по кассовой выручке
print(f"Топ знаменитых актёров по кассовой выручке: {len(celebr_actors)}")

top_persons = [actor_tuple[0] for actor_tuple, _ in freq_actors_overall] # топ знаменитостей по участию в фильмах
print(f"Топ знаменитостей по количеству участий в фильмах: {len(top_persons)}")

top_persons.extend(celebr_actors)
top_persons = set(top_persons)
print(f"Суммированное количество уникальных знаменитостей: {len(top_persons)}")
top_persons

In [ ]:
# 1. Количество актёров, задействованных в фильме.
train['num_cast'] = train['cast'].apply(lambda x: len(x) if x!=[] else 0)
test['num_cast'] = test['cast'].apply(lambda x: len(x) if x!=[] else 0)


# 2. Наличие в актёрском составе звёзд из списка топ-знаменитостей, а также идентификация конкретных имён.
train_list_of_cast = train['cast'].apply(lambda x: [j['name'] for j in x if x!=[]])
test_list_of_cast = test['cast'].apply(lambda x: [j['name'] for j in x if x!=[]])

train = get_binary_features(top_persons, train, train_list_of_cast, "actor_")
test = get_binary_features(top_persons, test, test_list_of_cast, "actor_")


# 3. Распределение актёров по полу — доля мужчин и женщин в составе.
train_list_of_gender = train['cast'].apply(lambda x:[i['gender'] for i in x if x!=[]])
test_list_of_gender = test['cast'].apply(lambda x:[i['gender'] for i in x if x!=[]])

train['unknown/non-binary'] = train_list_of_gender.apply(lambda x: sum([1 for i in x if (i==0)|(i==3)]))
train['female'] = train_list_of_gender.apply(lambda x: sum([1 for i in x if i==1]))
train['male'] = train_list_of_gender.apply(lambda x: sum([1 for i in x if i==2]))

test['unknown/non-binary'] = test_list_of_gender.apply(lambda x: sum([1 for i in x if (i==0)|(i==3)]))
test['female'] = test_list_of_gender.apply(lambda x: sum([1 for i in x if i==1]))
test['male'] = test_list_of_gender.apply(lambda x: sum([1 for i in x if i==2]))


# Наличие имени главного героя в топ-списке главных героев
s = Counter([i['character'] for j in train['cast'] for i in j if i['order']==0]).most_common(21)
top_characters = [i[0] for i in s]
top_characters.remove('')

train_list_of_character = train['cast'].apply(lambda x: [i['character'] for i in x if (x!=[])&(i['order']==0)])
test_list_of_character = test['cast'].apply(lambda x: [i['character'] for i in x if (x!=[])&(i['order']==0)])

train = get_binary_features(top_characters, train, train_list_of_character, "character_")
test = get_binary_features(top_characters, test, test_list_of_character, "character_")


# Дополним список фичей для удаления
columns_to_drop.append("cast")
columns_to_drop

### Feature - **crew**

**crew** - это сведения о всей съёмочной группе фильма — специалистах, работавших за кадром. В нём хранится список людей с указанием их ролей в производстве, включая режиссёров, продюсеров, сценаристов, операторов, монтажёров, композиторов, художников по костюмам, звукорежиссёров и других участников. Каждый элемент описывает одного человека и включает его имя, уникальный идентификатор, должность в проекте, отдел (например, режиссура, производство, звук), а также дополнительные сведения, такие как популярность и ссылка на профильное изображение.

In [ ]:
train['crew'][0][:7]

Полагаю, что атрибуты department и job представляют одну и ту же информацию, только второй более подробнее описывает деятельность члена съёмочной группы.
Проанализируем атрибуты - **department** and **job**.

In [ ]:
# Число уникальных депертаментов 
departments = [i['department'] for crew in train['crew'] for i in crew]
top_departments = Counter(departments).most_common()
print(f"Количество уникальных департаментов всего в датасете: {len(top_departments)}\n")

# Посмотрим на топ-15 департаментов
top_departments = [i[0] for i in top_departments[:15]]
display(top_departments)



print("\n")
# Число уникальных профессий
train_list_of_job = [i['job'] for j in train['crew'] if j!=[] for i in j]
freq_jobs = Counter(train_list_of_job).most_common()
print(f"Общее число наименований профессий: {len(freq_jobs)}")

# Посмотрим на топ 15 профессий
top_jobs = [i[0] for i in freq_jobs[:15]]
top_jobs

| 🎬 Профессия                                                  | 📖 Описание                                                                                 |
| ------------------------------------------------------------- | ------------------------------------------------------------------------------------------- |
| **Producer (Продюсер)**                                       | Организует производство фильма: финансирование, команда, контроль съёмочного процесса.      |
| **Executive Producer (Исполнительный продюсер)**              | Отвечает за финансовую и административную сторону, обеспечивает запуск проекта.             |
| **Director (Режиссёр)**                                       | Главный творческий руководитель: определяет стиль, работает с актёрами и съёмочной группой. |
| **Screenplay (Сценарист)**                                    | Пишет сценарий: сюжет, сцены, диалоги.                                                      |
| **Editor (Монтажёр)**                                         | Собирает отснятый материал в финальный фильм, задаёт ритм повествования.                    |
| **Casting (Кастинг-директор)**                                | Подбирает актёров на роли, организует пробы.                                                |
| **Director of Photography (Оператор-постановщик)**            | Отвечает за визуальный стиль, свет, кадры и композицию.                                     |
| **Original Music Composer (Композитор)**                      | Создаёт оригинальную музыку к фильму.                                                       |
| **Art Direction (Художественный руководитель)**               | Разрабатывает визуальную концепцию и художественное оформление сцен.                        |
| **Production Design (Постановщик-производства)**              | Определяет внешний вид фильма: декорации, интерьеры, локации.                               |
| **Costume Design (Художник по костюмам)**                     | Создаёт костюмы, отражающие характеры и эпоху персонажей.                                   |
| **Writer (Писатель)**                                         | Автор литературного материала, на основе которого создаётся сценарий.                       |
| **Set Decoration (Декоратор площадки)**                       | Оформляет съёмочную площадку, подбирает реквизит и детали интерьера.                        |
| **Makeup Artist (Гримёр)**                                    | Делает макияж и грим, включая спецэффекты.                                                  |
| **Sound Re-Recording Mixer (Звукорежиссёр финального микса)** | Сводит все звуковые дорожки — речь, музыку и эффекты — в единый финальный звук.             |


#### Top crew members by popular jobs

In [ ]:
# pd.reset_option("display.max_colwidth")
top_popular_members = [(i['name'], i['profile_path'], i['id']) for crew_info in train['crew'] for i in crew_info if i['job'] in top_jobs]
top_popular_members = Counter(top_popular_members).most_common()
print(f"Число людей съёмочной группы топ популярных фильмов по выручке: {len(top_popular_members)}")

top_popular_members = top_popular_members[:20]
top_popular_members

In [ ]:
# Соберем необходимые данные знаменитостей за кадром в датафрейм
top_popular_members_df = pd.DataFrame({
    "name": [i[0] for i, freq in top_popular_members],
    "profile_path": [i[1] for i, freq in top_popular_members ],
    "id": [i[2] for i, freq in top_popular_members ],
})

# добавим их профессии
top_popular_members_df.loc[:, 'jobs'] = top_popular_members_df['name'].apply(
            lambda x: list(set([i['job'] for j in top_films['crew'] for i in j if i['name']==x])))

print(f"Топ знаменитостей за кадром - {len(top_popular_members_df)}")
top_popular_members_df

Обновим сразу постеры топ персон съёмочных групп, а также заполним пустые списки профессий наименованием департамента из источника по id персоны "https://api.themoviedb.org/3/person/"

In [ ]:
url = "https://api.themoviedb.org/3/person/{}"
params = {
    "api_key": API_KEY,
    "language": "en-US"
}

for i, person_info in enumerate(top_popular_members_df.values):
    name, path, person_id, jobs = person_info
    response = requests.get(url.format(person_id), params=params)
    if response.status_code==200:
        result = response.json()
    else:
        print(f"Ошибка: {response.status_code}, {response.text}")
    new_profile_path = result.get('profile_path')
    if jobs == []:
        new_jobs = result.get("known_for_department")
        top_popular_members_df.loc[top_popular_members_df['name']==name, "jobs"] = new_jobs
    top_popular_members_df.loc[top_popular_members_df['name']==name, "profile_path"] = new_profile_path

top_popular_members_df

Посмотрим на топ знаменитостей за кадром.

In [ ]:
fig = plt.figure(figsize=(40, 40))

for i, person_info in enumerate(top_popular_members_df.values):
    name, path, person_id, jobs = person_info

    if not path:
        continue
    ax = fig.add_subplot(5,4, i+1, xticks=[], yticks=[])
    try:
        im = Image.open(urlopen(f"https://image.tmdb.org/t/p/w500{path}"))
        ax.imshow(im)
        # поработаем над отображением профессий чтобы список не занимал много места по ширине
        if not isinstance(jobs, list):
            title_jobs = jobs
        else:
            title_jobs = "\n".join(jobs)
        ax.set_title(
            (f"Name: {name} \n jobs: {title_jobs}"),
             fontdict={
            "fontsize": 40,
            "fontname": "Arial",
            "color": "#990F02"    
            }
                    )
        sleep=(0.2)
    except Exception as e:
        print("Ошибка: ", e)
plt.tight_layout()
plt.show();

#### gender distribution

In [ ]:
list_of_gender = train['crew'].apply(lambda x:[i['gender'] for i in x if x!={}])
s = [gender for i in list_of_gender for gender in i]
Counter(s).most_common()

In [ ]:
# Список персон в составе съёмочных групп относящих себя к гендеру Non-binary
gender_3 = [(i['name'], i['profile_path']) for j in train['crew']  for i in j if i['gender']==3]
# Подсчёт количества уникальных актёров с гендерным идентификатором Non-binary
unique_gender_3 = Counter(i for i in gender_3).most_common()
print(f"Количество персон с гендерным идентификатором - Non-binary в составе съёмочных групп в целом по базе TMDB:{len(unique_gender_3)}")


# Список фильмов, в съёмочной группе которого есть член группы  с гендерным идентификатором Non-binary
gender_3_order_0 = [(i['name'], i['profile_path']) for j in train['crew']  for i in j if i['gender']==3]
print(f"Количество фильмов, в съёмочной группе которого есть член группы с гендерным признаком None binary: {len(gender_3_order_0)}")

unique_gender_3[:20]

In [ ]:
# Список персон в составе съёмочных групп относящих себя к гендеру Non-binary
gender_3 = [(i['name'], i['profile_path']) for j in train['crew']  for i in j if i['gender']==3]
# Подсчёт количества уникальных актёров с гендерным идентификатором Non-binary
unique_gender_3 = Counter(i for i in gender_3).most_common()
print(f"Количество персон с гендерным идентификатором - Non-binary в составе съёмочных групп в целом по базе TMDB:{len(unique_gender_3)}")


# Список фильмов, в съёмочной группе которого есть член группы  с гендерным идентификатором Non-binary
gender_3_order_0 = [(i['name'], i['profile_path']) for j in train['crew']  for i in j if i['gender']==3]
print(f"Количество фильмов, в съёмочной группе которого есть член группы с гендерным признаком None binary: {len(gender_3_order_0)}")

fig = plt.figure(figsize=(48, 40))  
for i, person_info in enumerate(unique_gender_3[:30]):
    info, freq = person_info
    name, path = info
    if not path:
        continue
    try:
        ax = fig.add_subplot(6, 5, i+1, xticks=[], yticks=[])  
        im = Image.open(urlopen(f"https://image.tmdb.org/t/p/w500{path}"))
        ax.imshow(im)
        ax.set_title(name, fontdict={
            "fontsize": 40,
            "fontname": "Arial",
            "fontweight": "bold",  
            "color": "#990F02"    
        })
    except Exception as e:
        print(f"Ошибка при запросе постера {i["name"]}; {e}")
        continue
    sleep=(0.2)
plt.tight_layout()
plt.show();

Из этих данных также как с признаком cast, извлечем следующие признаки:
1. Участвовал ли в съемках фильма кто нибудь из топ популярных персон
2. Гендерное распределение съёмочной группы, то  есть сколько человек каждого пола участвовали  в группе
3. Количество людей в съёмочной группе фильма
4. Распределение людей по спектру профессий в съёмочной группе, то есть сколько человек работали с каждого депертамента над фильмом (атрибут department)
5. Признак, отражающий наличие в съёмочной группе специалиста, чья профессия входит в число наиболее часто встречающихся. 

#### Feature engineering

In [ ]:
# 1. Участвовал ли в съемках фильма кто нибудь из топ популярных персон
top_popular_crew_members =  [members_info[0] for members_info, freq in top_popular_members]


train_list_of_crew =  train['crew'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) and len(x)>0  else ['no crew'] )
test_list_of_crew =  test['crew'].apply(lambda x: [i['name'] for i in x] if isinstance(x, list) and len(x)>0  else ['no crew'])

train = get_binary_features(top_popular_crew_members, train, train_list_of_crew, "crew_name_")
test = get_binary_features(top_popular_crew_members, test, test_list_of_crew, "crew_name_")



# 2. Гендерное распределение съёмочной группы, то  есть сколько человек каждого пола участвовали  в группе
train_list_genders_crew = train['crew'].apply(lambda x: [i['gender'] for i in x] if isinstance(x, list) and len(x)>0 else ['no crew'])
test_list_genders_crew = test['crew'].apply(lambda x: [i['gender'] for i in x] if isinstance(x, list) and len(x)>0 else ['no crew'])


train['unknown/non-binary_crew'] = train_list_genders_crew.apply(lambda x: sum([1 for i in x if (i==0)|(i==3)]))
train['female_crew'] = train_list_genders_crew.apply(lambda x: sum([1 for i in x if i==1]))
train['male_crew'] = train_list_genders_crew.apply(lambda x: sum([1 for i in x if i==2]))

test['unknown/non-binary_crew'] = test_list_genders_crew.apply(lambda x: sum([1 for i in x if (i==0)|(i==3)]))
test['female_crew'] = test_list_genders_crew.apply(lambda x: sum([1 for i in x if i==1]))
test['male_crew'] = test_list_genders_crew.apply(lambda x: sum([1 for i in x if i==2]))



# 3. Количество людей в съёмочной группе фильма
train['num_crew'] = train['crew'].apply(lambda x: len(x) if isinstance(x, list) and len(x) >0 else 0)
test['num_crew'] = test['crew'].apply(lambda x: len(x) if isinstance(x, list) and len(x) >0 else 0)



# 4. Распределение людей по спектру профессий в съёмочной группе, 
# то есть сколько человек работали с каждого депертамента над фильмом (атрибут department)
train_list_of_department = train['crew'].apply(lambda x: [i['department'] for i in x] if isinstance(x, list) and len(x)>0 else ['no crew'])
test_list_of_department = test['crew'].apply(lambda x: [i['department'] for i in x] if isinstance(x, list) and len(x)>0 else ['no crew'])

train = get_binary_features(top_departments, train, train_list_of_department, "depart_")
test = get_binary_features(top_departments, test, test_list_of_department, "depart_")



# 5. Признак, отражающий наличие в съёмочной группе специалиста, чья профессия входит в число наиболее часто встречающихся.
train_list_of_jobs = train['crew'].apply(lambda x: [i['job'] for i in x] if isinstance(x, list) and len(x) >0 else ['no crew'])
test_list_of_jobs = test['crew'].apply(lambda x: [i['job'] for i in x] if isinstance(x, list) and len(x) >0 else ['no crew'])

train = get_binary_features(top_jobs, train, train_list_of_jobs, "crew_job_")
test = get_binary_features(top_jobs, test, test_list_of_jobs, "crew_job_")


# Дополним список фичей для удаления
columns_to_drop.append("crew")
columns_to_drop

Удаляем все признаки, из которых уже извлекли полезые фичи.

In [ ]:
train = train.drop(columns=columns_to_drop)
test = test.drop(columns=columns_to_drop)

## EDA

In [ ]:
train.info()

Начнем рассматривать числовые признаки,  исключая те, что были созданы в процессе иследования предыдущих признаков, их всего 6:
'budget', 'runtime', 'popularity', 'vote_average', 'vote_count', 'revenue'.

Изучим их.

In [ ]:
train.select_dtypes(include=['int64', 'float64']).dtypes.head(7)

In [ ]:
train[['budget', 'runtime', 'popularity', 'vote_average', 'vote_count', 'revenue']]

#### budget

Рассмотрим взаимозависимость признака budget с целевым признаком и с логарифмом таргета

In [ ]:
fontdict = {"fontfamily": "arial", "fontsize":15, "fontweight": True, "color": "#682F2F"}
plt.figure(figsize=(15,12))

plt.subplot(3,2,1)
sns.boxenplot(train['revenue'],  color="#D6B2B1")
plt.title("Распределение выручки", fontdict=fontdict)

plt.subplot(3,2,2)
sns.boxenplot(np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Распределение логарифма выручки", fontdict=fontdict)

plt.subplot(3,2,3)
sns.boxenplot(train['budget'], color="#D6B2B1")
plt.title("Распределение бюджета", fontdict=fontdict)

plt.subplot(3,2,4)
sns.boxenplot(np.log1p(train['budget']), color="#D6B2B1")
plt.title("Распределение логарифма бюджета", fontdict=fontdict)

plt.subplot(3,2,5)
sns.scatterplot(data=train, x='budget', y = np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Бюджет и логарифмированная выручка", fontdict=fontdict)

plt.subplot(3,2,6)
sns.scatterplot(data=train, x=np.log1p(train['budget']), y = np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Логарифмированный бюджет и логарифмированная выручка", fontdict=fontdict)

plt.tight_layout()
plt.show();

Наблюдается некая корреляция между бюджетом и выручкой, особенно ярче выражается при логарифмировании оюоих признаков

#### popularity

In [ ]:
plt.figure(figsize=(15,8))
plt.subplot(2,2,1)
sns.boxenplot(train['popularity'],  color="#D6B2B1")
plt.title("Распределение рейтинга", 
          fontdict=fontdict)
plt.subplot(2,2,2)
sns.boxenplot(np.log1p(train['popularity']), color="#D6B2B1")
plt.title("Распределение лоагрифмированного рейтинга", 
          fontdict=fontdict)
plt.subplot(2,2,3)
sns.scatterplot(data=train, x='popularity', y = np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Рейтинг и логарифмированная выручка", 
          fontdict=fontdict)
plt.subplot(2,2,4)
sns.scatterplot(data=train, x=np.log1p(train['popularity']), y = np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Логарифмированный рейтинг и логарифмированная выручка", 
          fontdict=fontdict)
plt.tight_layout()
plt.show();

Как мы видим некая корреляция наблюдается при логарифмировании признака рейтинга и целевой метки.

#### runtime

In [ ]:
plt.figure(figsize=(15,8))
plt.subplot(2,2,1)
sns.boxenplot(train['runtime'],  color="#D6B2B1")
plt.title("Распределение длительности фильма", 
          fontdict=fontdict)
plt.subplot(2,2,2)
sns.boxenplot(np.log1p(train['runtime']), color="#D6B2B1")
plt.title("Распределение лоагрифмированной длительности фильма", 
          fontdict=fontdict)
plt.subplot(2,2,3)
sns.scatterplot(data=train, x='runtime', y = np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Длительность фильма и логарифмированная выручка", 
          fontdict=fontdict)
plt.subplot(2,2,4)
sns.scatterplot(data=train, x=np.log1p(train['runtime']), y = np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Логарифмированная длительность фильма и логарифмированная выручка", 
          fontdict=fontdict)
plt.tight_layout()
plt.show();

Корреляции между длительностью фильмов и выручкой не наблюдается, высокая выручка фильмов с длительностью 1.5 - 2 часа объсняется тем, что основная доля фильмов также по длительности находится в этом диапазоне. 

#### Features **vote_average** and **vote_count**

Атрибуты vote_average и vote_count — это два взаимосвязанных показателя, которые обычно встречаются в кино-датасетах. Они описывают оценку фильма зрителями и количество проголосовавших. Этот показатель не учитывает, сколько человек проголосовало.
То есть фильм с 10 голосами и 100 000 голосов может иметь одинаковое значение 8.0 — но достоверность у них разная.

Среднее может быть искажено при малом числе голосов (например, редкие или новые фильмы). Чем выше vote_count, тем надёжнее значение vote_average.



Чтобы учесть и оценку, и число голосов, можно применить взвешенную среднюю по IMDb-формуле:
$$
\text{Weighted Rating} = \frac{v}{v + m} \cdot R + \frac{m}{v + m} \cdot C
$$
где:

* 𝑅 — vote_average фильма,

* 𝑣 — vote_count фильма,

* 𝑚 — минимальное число голосов для попадания в рейтинг,

* 𝐶 — средний vote_average по всей выборке.

Это классическая формула IMDb.(Информация получена из ChatGPT)

In [ ]:
train['vote_count'].hist(color=["#CFA27E"])

In [ ]:
# Посмотрим базовую статистику 
print(train[['vote_average', 'vote_count']].describe())

In [ ]:
# Посчитаем среднее по всей базе и обозначим параметр m для расчета метрики по формуле

C = train['vote_average'].mean()     # средний рейтинг по всем фильмам
m = 4000 # минимальное число голосов для попадания в рейтинг, отобрано по распределению числа голосов


# IMDb-взвешенный рейтинг 
def weighted_rating(df, m=m, C=C):
    v = df['vote_count']
    R = df['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C) # IMDb-формула


train['weighted_vote_rating'] = weighted_rating(train)
test['weighted_vote_rating'] = weighted_rating(test)


# Удаляем исходные признаки
train = train.drop(columns=['vote_average', 'vote_count'])
test = test.drop(columns=['vote_average', 'vote_count'])

In [ ]:
fontdict = {"fontfamily": "arial", "fontsize":15, "fontweight": True, "color": "#682F2F"}
plt.figure(figsize=(15,6))

plt.subplot(1,2,1)
sns.boxenplot(train['weighted_vote_rating'],  color="#D6B2B1")
plt.title("Распределение среднего голосового рейтинга", fontdict=fontdict)


plt.subplot(1,2,2)
sns.scatterplot(data=train, x='weighted_vote_rating', y = np.log1p(train['revenue']), color="#D6B2B1")
plt.title("Средний голосовой рейтинг и логарифмированная выручка", fontdict=fontdict)


plt.tight_layout()
plt.show();

кажется выручка от полученной  метрики незначительно зависит. 

#### Release date by years 

In [ ]:
d1 = train['release_date_year'].value_counts().sort_index()
d2 = test['release_date_year'].value_counts().sort_index()

fig, ax = plt.subplots(1,1, figsize=(12,6))

sns.lineplot(x=d1.index, y=d1.values, marker='o', color='#9E726F', label='Train', ax=ax)
sns.lineplot(x=d2.index, y=d2.values, marker='o', color='#F3AB60',  label='Test', ax=ax)

plt.title('Количество фильмов по годам')
plt.xlabel('Год')
plt.ylabel('Количество фильмов')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show();

На графике отчётливо видно, что к 2020 году наблюдается резкий спад в количестве выпускаемых фильмов, за которым следует стремительный рост.
Такое поведение связано с периодом пандемии COVID-19: в это время действовали строгие ограничения на массовые мероприятия, а кинотеатры и съёмочные площадки были закрыты.
После окончания карантина, когда ограничения были сняты, зрители, соскучившиеся по кино и большим премьерам, вновь активно стали посещать кинотеатры, что привело к заметному росту спроса и числа выпускаемых фильмов.   

Посмотрим на тренд выручки и количество выпускаемых фильмов на общем датасете, то есть в данных до разделения на обучающую и тестовую выборки.

In [ ]:
# Заполним пропуск, мы знаем что в поле даты выпуска фильмов всего один пропуск
data['release_date'] = data['release_date'].fillna(release_date, axis=0)
data['release_date'].isna().sum()

Произведем те же преобразования с полем даты выпуска фильмов, что и с полем разделенных выборок.

In [ ]:
data_for_analyze = date_decomposition(data)

Визуализируем зависимость между двумя показателями: число выпускаемых фильмов и средняя выручка по годам.

In [ ]:
plt.figure(figsize=(12, 8))
d1 = data['release_date_year'].value_counts().sort_index()
d2 = data.groupby(['release_date_year'])['revenue'].mean()

data_for_layout = [
    go.Scatter(
        x=d1.index, 
        y=d1.values, 
        name='Film count',
        line=dict(color='#9E726F'),  
        mode='lines+markers'         
    ),
    go.Scatter(
        x=d2.index, 
        y=d2.values, 
        name='Mean revenue',
        yaxis='y2',
        line=dict(color='#F3AB60', dash='solid'),
        mode='lines+markers'
    )
]

layout = go.Layout(
    title="Количество фильмов и средняя выручка по годам",
    xaxis=dict(title='Year'),
    yaxis=dict(title='Count'),
    yaxis2=dict(title='Mean revenue', overlaying='y', side='right'),
    legend=dict(orientation="h")
)

py.iplot(dict(data=data_for_layout, layout=layout));

Можно отметить интересную динамику средней выручки: после резкого спада в период пандемии, несмотря на последующий стремительный рост числа выпускаемых фильмов, средняя выручка так и не восстановилась до доковидного уровня. В 2021 году наблюдался кратковременный рост, однако в последующие годы — вплоть до 2024 — показатель стабильно снижался. К 2025 году выручка начала постепенно восстанавливаться, хотя количество выпущенных фильмов, наоборот, уменьшилось. Вероятно, это связано с тем, что год ещё не завершён, и общий тренд может со временем выровняться.

Это может быть связано с несколькими весомыми причинами:
* Изменение структуры кинорынка: многие студии переключились на онлайн-премьеры и стриминговые платформы, многие фильмы стали минуя кинотеатры выходить сразу онлайн или с коротким театральным окном (2–3 недели вместо прежних 3–6 месяцев).
* Рост числа малобюджетных фильмов. После пандемии киностудии стремились снизить риски — стали снимать больше дешёвых или независимых фильмов, чтобы быстрее окупить производство.
Из-за этого общее количество релизов выросло, но средняя выручка (на один фильм) уменьшилась.
* Изменение поведения зрителей. Многие зрители не вернулись в кино в прежнем объёме — часть аудитории осталась на онлайн-просмотрах.
Кроме того, кино стало восприниматься как «особое событие» — люди идут в кино только на крупные премьеры (Marvel, Dune, Avatar и т.п.), а не на каждый новый фильм.
* После пандемии в разных странах наблюдался рост цен и снижение доходов населения.
Это тоже повлияло на спрос на офлайн-развлечения, включая походы в кино.
* Конкуренция и переизбыток контента. После снятия ограничений в 2022–2023 годах на рынок обрушился поток отложенных премьер — фильмов стало слишком много. В итоге зрительская аудитория и кассовые сборы размазались по множеству релизов, снижая среднюю выручку на каждый.


Здесь не сложно заметить, что т.к. из обучающей выборки полностью исключен 2025 год, модель сорее всего плохо будет предсказывать выручку фильмов последнего года.

Однако в реальных задачах такая ситуация вполне типична — ведь нередко требуется делать прогнозы именно на будущее.

#### Release date by weeks

In [ ]:
sns.catplot(x='release_date_weekday', y='revenue', data=train, palette=palette);
plt.title('Revenue on different days of week of release');

Кажется фильмы, выпущенные во вторник,  среду и четверг, как правило, имеют более высокий доход.

#### Release date by quarters

In [ ]:
sns.catplot(x='release_date_quarter', y='revenue', data=train, palette=palette)
plt.title('Revenue on different days of quarter of release');

Также по квартальному распределению можно заметить, что фильмы, выпущенные во втором и четвёртом кварталах, оказываются наиболее кассовыми. Это может быть связано с тем, что четвёртый квартал приходится на праздничный сезон — предновогодние и рождественские месяцы, когда повышается интерес к развлечениям, а второй квартал совпадает с весенним периодом, когда аудитория постепенно возвращается к активному отдыху после зимних праздников.

#### belongs_to_collection & homepage

In [ ]:
plt.figure(figsize=(10,10))
plt.subplot(2,1,1)
color_list = ['#9F8A78', '#F3AB60']
sns.boxenplot(data=train, x='has_collection', y='revenue', palette=["#9F8A78", "#F3AB60"])
plt.title("Распределение выручки в зависимости от принадлежности к коллекции", fontdict={"fontfamily": "arial","color": "#682F2F"})
plt.subplot(2,1,2)
color_list = ['#9E726F', '#B9C0C9']
sns.boxenplot(data=train, x='has_homepage', y='revenue', palette=["#9F8A78", "#F3AB60"])
plt.title("Распределение выручки в зависимости от наличия официального сайта", fontdict={"fontfamily": "arial","color": "#682F2F"})
plt.tight_layout()
plt.show();

#### genres

In [ ]:
genres_features = [i for i in train.columns if ("genre") in (i)]
print("Все признаки, извлеченные из данных genres:\n")
pprint(genres_features)

##### NumOfGenres

In [ ]:
x = train['NumOfGenres'].value_counts().index
print(f"\nЧисло уникальных значений признка numofgenres: {len(x)}\n")
sns.catplot(data=train, y='revenue', x='NumOfGenres', palette=palette[:8], height=5, aspect=1.5);
plt.title("Количество жанров и выручка", fontdict=fontdict)
plt.show();

Похоже, фильмы, относящиеся к 3–4 жанрам, приносят наибольшую выручку — меньшее или, наоборот, слишком большое количество жанров оказывает менее положительный эффект.

##### genres-features

In [ ]:
genres_features = [i for i in train.columns if ("genre_") in (i)]
print("Остальные признаки, извлеченные из данных genres:\n")
pprint(genres_features)
n = len(genres_features)
print(f"\nИх всего: {n}")

In [ ]:
fig, axes = plt.subplots(8, 2, figsize=(16, 48))  # 2*8=16, 8*6=48

for i, col in enumerate(genres_features):
    ax = axes[i // 2, i % 2] 
    sns.boxenplot(
        data=train,
        x=col,
        y = np.log1p(train['revenue']),
        palette=["#CFA27E", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
    

plt.tight_layout()
plt.show();

Таким образом, наибольшую выручку приносят фильмы жанров приключений и боевиков, а также неплохие результаты показывают жанры научной фантастики, фэнтези, аниме, семейные картины и исторические фильмы.
Наименее прибыльными оказались фильмы жанров войны, мюзикла, мистики, хоррора и документальные картины.

#### production_companies   

In [ ]:
prod_comp_features = [i for i in train.columns if "compan" in i] 
print("Все признаки, извлеченные из данных production_companies:\n")
pprint(prod_comp_features)

n = len(prod_comp_features)
print(f"\nИх всего: {n}")

In [ ]:
x = train['NumOfCompanies'].value_counts().index
print(f"\nЧисло уникальных значений признка numofcompanies: {len(x)}\n")
sns.catplot(data=train, y='revenue', x='NumOfCompanies', palette=palette, height=6, aspect=2);
plt.title("Количество кинокомпаний и выручка", fontdict=fontdict)
plt.show();

Наибольшую выручку приносят фильмы, созданные при участии 1–3 кинокомпаний, в то же время проекты, с 4-5 или более кинокомпаниями показывают меньшую доходность.

In [ ]:
prod_companies = [i for i in train.columns if "prod_comp" in i] 
print("Общее число топ студий:", len(prod_companies))
fig, axes = plt.subplots(20, 2, figsize=(18, 100))  # 3*6=18, 10*4=40

for i, col in enumerate(prod_companies):
    ax = axes[i // 2, i % 2] 
    sns.boxenplot(
        data=train[train['revenue']<=1500000000],
        x=col,
        y='revenue', #np.log1p(train['revenue']),
        palette=["#D6B2B1", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.show();

На графиках видно, что наиболее высокодоходные фильмы выпускает "Universal Pictures" — значительная доля выручки этой кинокомпании, за исключением выбросов,  достигает 1 млрд долларов.

У компаний "Warner Bros. Pictures",  "Columbia Pictures", "Twentieth Century Fox Film Corporation" этот показатель составляет около 770 млн.

Фильмы, выпущенные "TSG Entertainment", "Paramount Pictures" и "Metro-Goldwyn-Mayer", также можно отнести к числу прибыльных — их основная выручка находится в диапазоне 600 млн долларов.

Что касается остальных кинокомпаний, то " а у "New Line Cinema" и "China Film Group Corporation" демонстрируют результаты на уровне 400-450 млн,
а выручка прочих студий не превышает 200 млн долларов.

#### production_countries

In [ ]:
prod_countries_features = [i for i in train.columns if "countr" in i]
print(f"Количество признаков извлеченных из данных по странам: {len(prod_countries_features)}")
prod_countries_features

In [ ]:
sns.catplot(data=train, y='revenue', x='NumOfCountries', palette=palette, height=6, aspect=2);
plt.title("Количество стран, задействованных  в производстве фильма и его выручка", fontdict=fontdict)
plt.show();

Кажется, в производстве фильмов с наиболее высокой выручкой участвовали одна или две страны, при этом чем больше количество стран задействовано, тем ниже доход от фильма. 

Возможные причины такого тренда:
1. Крупнейшие кассовые хиты — в основном американские. Подавляющее большинство блокбастеров (Marvel, Universal, Warner Bros.) — это продукция одной страны — США. США обладают мощной индустрией: огромные бюджеты, маркетинг, глобальная дистрибуция. Таким образом, даже при одной стране фильм может быть рассчитан на весь мировой рынок. Поэтому высокая выручка чаще связана с доминированием одной сильной киностраны, а не количеством участников.

2. Многонациональные фильмы часто — артхаус или нишевые
Когда в производстве участвуют 3–5 стран, это обычно не блокбастеры, а европейские драмы, фестивальные картины, документалки.
Они получают финансирование из разных стран (софинансирование), но:имеют ограниченный релиз, не ориентированы на массового зрителя, редко выходят в широкий международный прокат. В итоге — высокое качество, культурная ценность, но низкие кассовые сборы.

3. Много стран = сложная координация и компромиссы. Копродукции требуют согласования интересов разных продюсеров, языков, культурных нюансов. Это может удлинять производство, повышать издержки, снижать целостность творческой концепции.
Иногда это негативно отражается на коммерческом успехе.

##### Выручка в зависимости от страны - производителя.

In [ ]:
prod_countries = [i for i in train.columns if "prod_country" in i]
print("Общее число топ стран:", len(prod_countries))


fig, axes = plt.subplots(13, 2, figsize=(12, 52))  # 3*6=18, 10*4=40
fontdict_title = {'fontfamily': 'arial', 'color': '#682F2F', 'fontsize': 50, 'fontweight': True}

for i, col in enumerate(prod_countries):
    ax = axes[i // 2, i % 2] 
    sns.boxenplot(
        data=train[train['revenue']<=1500000000], # исключаем выбросы
        x=col,
        y='revenue',                  #np.log1p(train['revenue']),
        palette=["#D6B2B1", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)

fig.suptitle(
    "Выручка в зависимости от страны-производителя",
    fontfamily='Arial',
    color='#682F2F',
    fontsize=20,
    fontweight='bold',
    y=1.0
)
plt.tight_layout()
plt.show();

Как и ожидалось, наибольшие доходы приносят фильмы, созданные в США  — их выручка достигает 1 млрд долларов, за исключением выбросов, выручки которых достигли 1,5 млрд долларов. А также фильмы разработанные в Великобритании и Франции можно отнести к топ кассовым фильмам, основная доля их выручки достигает до 800 млн длр.  




Фильмы, выпущенные в Германии и Новой Зеландии, зарабатывают до 600 млн долларов. Среди остальных стран выделяется Канада, где фильмы приносят выручку до 250 млн долларов, тогда как в других странах этот показатель, как правило, не превышает 200 млн.Однако начиная сконца 2024 года, есть и такие страны как Китай и Венгрия среди лидирующих стран по кассовой выручке.

#### spoken languages

In [ ]:
top_languages_features = [i for i in train.columns if "lang" in i]
top_languages_features

In [ ]:
sns.catplot(data=train, y='revenue', x='NumOfLanguages', palette=palette, height=6, aspect=2);
plt.title("Количество языков, звучащие в фильме и выручка", fontdict=fontdict)
plt.show();

Похоже, что самые высокодоходные фильмы — это те, в которых преимущественно говорят на одном или двух языках.

In [ ]:
sns.catplot(data=train, y='revenue', x='original_language', palette=palette, height=6, aspect=2);
plt.title("Язык, на котором фильм изначально был создан и снят и выручка", fontdict=fontdict)
plt.xticks(train['original_language'], rotation=90)
plt.show();

In [ ]:
data[(data['original_language'] == 'zh')&(data['revenue']>2000000000)]['title']

Здесь наблюдается явный контраст: фильмы, снятые на английском языке, превосходят по выручке с большим отрывом, относительно картин, снятых на других языках. А также выделяется высокой выручкой фильмы снятые на мандаринском - Китайском языке.

In [ ]:
spok_langs = [i for i in train.columns if "spok" in i]
print("Общее число топ языков:", len(spok_langs))

In [ ]:
spok_langs = [i for i in train.columns if "spok" in i]
print("Общее число топ языков:", len(spok_langs))


fig, axes = plt.subplots(13, 2, figsize=(12, 52))  # 2*6=18, 10*4=40
fontdict_title = {'fontfamily': 'arial', 'color': '#682F2F', 'fontsize': 50, 'fontweight': True}

for i, col in enumerate(spok_langs):
    ax = axes[i // 2, i % 2] 
    sns.boxenplot(
        data=train,
        x=col,
        y=np.log1p(train['revenue']),
        palette=["#D6B2B1", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)

fig.suptitle(
    "Логарифмированная выручка в зависимости от языка, который звучит в фильме",
    fontfamily='Arial',
    color='#682F2F',
    fontsize=20,
    fontweight='bold',
    y=1.0
)
plt.tight_layout()
plt.show();

По языковому признаку лидируют фильмы, снятые на английском языке.
Следом по уровню выручки идут картины на испанском, французском и китайском (мандаринском) языках.

### Jobs

In [ ]:
job_columns = [x for x in train.columns if "job" in x]
job_columns

In [ ]:
fig, axes = plt.subplots(8, 2, figsize=(16, 48))  # 2*8=16, 8*6=48

for i, col in enumerate(job_columns):
    ax = axes[i // 2, i % 2] 
    sns.violinplot(
        data=train,
        x=col,
        y = np.log1p(train['revenue']),   
        palette=["#CFA27E", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и логарифмированная выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
    

plt.tight_layout()
plt.show();

In [ ]:
train['crew_job_Writer'].mean()

В целом, наличие большинства профессий оказывает схожее положительное влияние на выручку — каждая из них добавляет в среднем несколько сотен миллионов.
Однако выделяются некоторые исключения: отсутствие **Режиссёра** заметно снижает кассовые сборы, тогда как наличие **Автора литературного источника** почти не повышает выручку и иногда даже уменьшает её.
Также значимое влияние имеют **Продюсер**, **Монтажёр**, **Композитор** и **Оператор-постановщик** — без их участия фильмы, как правило, зарабатывают меньше.

### Title

In [ ]:
title_cols = [i for i in train.columns if "title" in i]
train[title_cols]

Столбец title содержит перевод названия фильмов на английский, но есть исключения, некоторые фильмы не переведены, из названия в этом признаке остались в исходном виде

In [ ]:
plt.figure(figsize = (10, 10))
text = ' '.join(train['title'].values)
text = text.replace(': The Movie', '')
wordcloud = WordCloud(max_font_size=None, background_color='white', width=1200, height=1000, colormap=cmap).generate(text)
plt.imshow(wordcloud)
plt.title('Top words in titles')
plt.axis("off")
plt.show();

In [ ]:
plt.figure(figsize = (10, 10))
text = ' '.join(train['overview'].fillna('').values)
wordcloud = WordCloud(max_font_size=None, background_color='white', width=1200, height=1000, colormap=cmap).generate(text)
plt.imshow(wordcloud)
plt.title('Top words in overview')
plt.axis("off")
plt.show()

Попробуем построить линейню модель с фичами из весов слов в описании фильмов. 

### Tokenization of feature Oveview

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(min_df=5, 
                             stop_words='english',
                            max_features=5000,
                            norm='l2')

oveview_text_cols = vectorizer.fit_transform(train['overview'].fillna(''))

In [ ]:
train['log_revenue'] = np.log1p(train['revenue'])
test['log_revenue'] = np.log1p(test['revenue'])

linereg = LinearRegression()
linereg.fit(oveview_text_cols, train['log_revenue'])

In [ ]:
y_log_pred = linereg.predict(oveview_text_cols)
y_pred = np.expm1(y_log_pred)
y_pred

#### Проверка распределения значений признаков

In [ ]:
sns.kdeplot(oveview_text_cols.data, color='#3F5E64')
plt.title("Распределение TF-IDF весов", fontdict=fontdict)
plt.show();

Так как наш датасет это результаты векторизатора TfidfVectorizer(), то его значения очень близки к нулю или очень маленькие, MAPE может показать нестабильные значения.
В таком случае лучше использовать SMAPE, напишем формулу этой метрики и использем его тоже в числе метрик.

In [ ]:
def smape(y_true, y_pred):
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred))
    )

In [ ]:
print('MAE:', round(mae(train['revenue'], y_pred),2))
print("MAPE: ", round(mape(train['revenue'], y_pred), 2))
print("SMAPE: ", round(smape(train['revenue'], y_pred),2))

Итак, можно отметить, что признаки, сформированные с помощью векторизатора, не обладают предсказательной силой — качество модели по соответствующим метрикам оказалось низким. Следовательно, из текстового описания фильмов не удалось извлечь информативные признаки, поэтому этот столбец целесообразно удалить из данных.

### Tagline

In [ ]:
plt.figure(figsize = (10, 10))
text = ' '.join(train['tagline'].fillna('').values)
wordcloud = WordCloud(max_font_size=None, background_color='white', width=1200, height=1000, colormap=cmap).generate(text)
plt.imshow(wordcloud)
plt.title('Top words in overview')
plt.axis("off")
plt.show();

Проверим признаки с текстовыми значениями на пропуски

In [ ]:
text_columns = ["tagline", "original_title", "title", "overview"]
for col in text_columns:
    percent = train[col].isna().mean()
    print(col, " - ", percent)
    print()

Поскольку признак с описанием фильма не продемонстрировал предсказательной силы, можно предположить, что и другие текстовые признаки — такие как "keywords", "tagline", "original_title" и "title", содержащие ещё меньше информации о содержании фильмов, — также не внесут значимого вклада в качество модели. Поэтому их целесообразно удалить из данных.
Но прежде получим признаки по количеству слов в каждом признаке, возможно такие признаки внесут какой-то вклад в информативную нагрузку модели.

In [ ]:
for col in ['title', 'tagline', 'overview', 'original_title']:
    train['len_' + col] = train[col].fillna('').apply(lambda x: len(str(x)))
    train['words_' + col] = train[col].fillna('').apply(lambda x: len(str(x.split(' '))))

    test['len_' + col] = test[col].fillna('').apply(lambda x: len(str(x)))
    test['words_' + col] = test[col].fillna('').apply(lambda x: len(str(x.split(' '))))


### Status

In [ ]:
train['status'].value_counts()

In [ ]:
test['status'].value_counts()

Так как признак статус содержит практически одно значение, за исключением двух фильмов в тестовой выборке, этот признак также подлежит удалению.

In [ ]:
# Создаём новый список удаляемых признаков
drop_columns = text_columns
drop_columns.append('status')
drop_columns

### keywords

In [ ]:
keywords_columns = [x for x in train.columns if "keyword" in x]
print(len(keywords_columns))
keywords_columns

In [ ]:
fig, axes = plt.subplots(15, 2, figsize=(16, 90))  # 2*8=16, 15*6=90

for i, col in enumerate(keywords_columns):
    ax = axes[i // 2, i % 2] 
    sns.boxenplot(
        data=train,
        x=col,
        y = np.log1p(train['revenue']),   
        palette=["#CFA27E", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и логарифмированная выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
    

plt.tight_layout()
plt.show();

По результатам анализа топ-ключевых слов можно отметить, что фильмы, содержащие такие ключевые слова, как **duringcreditsstinger** и **aftercreditsstinger**, демонстрируют наиболее высокие кассовые сборы. Аналогично, фильмы с тегами **superhero**, **sequel**, **based on comic** и **remake** также относятся к числу наиболее прибыльных.
В то же время фильмы с ключевыми словами, связанными с темой **совершеннолетия**, не оказывают заметного влияния на общую выручку. Напротив, средние кассовые сборы фильмов с ключевыми словами **gay theme** и **lgbt** оказываются относительно ниже.


**duringcreditsstinger** - означает: Наличие сцены во время титров — то есть дополнительный эпизод, который показывают в процессе финальных титров, но до их окончания (как, например, в фильмах Marvel).

**aftercreditsstinger** — сцена после титров, уже в самом конце.

### Cast

In [ ]:
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.scatter(x=train['num_cast'], y=train['revenue'], color=cmap.colors[1])
plt.title('Количество актеров и выручка', fontdict=fontdict);
plt.subplot(1, 2, 2)
plt.scatter(train['num_cast'], train['log_revenue'], color=cmap.colors[1])
plt.title('Количество актеров и логарифмированная выручка', fontdict=fontdict);

In [ ]:
cast_columns = [x for x in train.columns if "actor" in x]
print(len(cast_columns))

In [ ]:
fig, axes = plt.subplots(17, 3, figsize=(18, 102))  # 3*6=18, 17*6=102

for i, col in enumerate(cast_columns):
    ax = axes[i // 3, i % 3] 
    sns.boxenplot(
        data=train,
        x=col,
        y = np.log1p(train['revenue']),   
        palette=["#CFA27E", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и логарифмированная выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
    

plt.tight_layout()
plt.show();

### Crew

In [ ]:
train['num_crew']

In [ ]:
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.scatter(x=train['num_crew'], y=train['revenue'], color=cmap.colors[-2])
plt.title('Количество людей в съёмочной группе и выручка', fontdict=fontdict);
plt.subplot(1, 2, 2)
plt.scatter(train['num_crew'], train['log_revenue'], color=cmap.colors[-2])
plt.title('Количество людей в съёмочной группе\n и логарифмированная выручка', fontdict=fontdict);

In [ ]:
crew_columns = [x for x in train.columns if "crew_name" in x]
print(len(crew_columns))
crew_columns

In [ ]:
fig, axes = plt.subplots(10, 2, figsize=(12, 60))  # 2*6=12, 10*6=60

for i, col in enumerate(crew_columns):
    ax = axes[i // 2, i % 2]
    sns.boxenplot(
        data=train,
        x=col,
        y = np.log1p(train['revenue']),   
        palette=["#CFA27E", "#B2D3A8"],
        ax=ax
    )
    ax.set_title(f"{col} и логарифмированная выручка", fontdict=fontdict)
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)
    

plt.tight_layout()
plt.show();

### Removal of unnecessary features

In [ ]:
train = train.drop(columns=drop_columns)
test = test.drop(columns=drop_columns)

## Target Revenue

In [ ]:
train.describe()

In [ ]:
test.describe()

В данных есть фильмы с выручкой ниже 10, например 1,2,3,... На самом деле выручка ниже 100$ , уже кажется аномальным или некорректным значением.

Посмотрим, что это за фильмы

In [ ]:
too_low_revenue_id = data[data['revenue']<100]['id']
data[data['id'].isin(too_low_revenue_id)][['id', 'budget', 'revenue', 'production_countries', 'genres', 'release_date', 'title', 'original_title']].sort_values(by='release_date').head(10)

In [ ]:
data[data.id==1564093][['id', 'runtime', 'homepage', 'title', 'production_countries', 'budget', 'revenue', 'crew']]

Фильмы с крайне низкими значениями бюджета и выручки, вероятнее всего, представляют собой короткометражные проекты, созданные фанатами, блогерами или независимыми энтузиастами и размещённые, например, на YouTube.

Проверим процент таких фильмов, где выручка больше 1000

In [ ]:
anomalous_revenue = data.id[data.budget>=1000][data.revenue <=100]
print(f"Процент фильмов с аномально низким значением выручки, при бюджете выше 1000 : {round(len(anomalous_revenue)/data.shape[0]*100, 2)}%")

In [ ]:
data[data['id'].isin(anomalous_revenue)][['id', 'budget', 'revenue', 'production_countries', 'genres', 'release_date', 'title', 'original_title']].sort_values(by='release_date').head(5)

In [ ]:
data[data.id==198701][['id', 'runtime', 'homepage', 'genres', 'title', 'production_countries', 'budget', 'revenue', 'crew']]

Официальных разъяснений от TMDb по поводу очень малых положительных значений в поле revenue (1, 2, 3 и т.д.) я не обнаружила. Если при этом и бюджет фильма также низкий — это ещё можно объяснить, однако для фильмов с бюджетом свыше 1000 такие значения выручки, вероятнее всего, являются ошибочными. Поэтому я воспользовалась подходом, предложенным одним из грандмастеров Kaggle https://www.kaggle.com/code/artgor/eda-feature-engineering-and-model-interpretation , и умножила эти значения на 1 000 000.

In [ ]:
# Train data
anom_rev_train = train.id[train.budget > 1000][train.revenue < 100]
for k in anom_rev_train :
    train.loc[train['id'] == k,'revenue'] =  train.loc[train['id'] == k,'revenue'] * 1000000

# Test data
anom_rev_test = test.id[test.budget > 1000][test.revenue < 100]
for k in anom_rev_test :
    test.loc[test['id'] == k,'revenue'] =  test.loc[test['id'] == k,'revenue'] * 1000000

Поскольку для обучения будут использоваться модели на основе деревьев решений — такие как LightGBM, CatBoost, XGBoost и другие, — которые инвариантны к масштабу признаков, данные не будут подвергаться очищению от выбросов и масштабированию.

## Feature engineering

Ниже приведены идеи по расширению пространства признаков за счёт генерации новых признаков путём арифметических преобразований.
Идеи и методы получены из ядра [Andrew Lukyanenko](https://www.kaggle.com/code/artgor/eda-feature-engineering-and-model-interpretation)

In [ ]:
def new_features(df):
    df['budget_to_popularity'] = df['budget'] / df['popularity']
    df['budget_to_runtime'] = df['budget'] / df['runtime']
    
    # some features from https://www.kaggle.com/somang1418/happy-valentines-day-and-keep-kaggling-3
    df['budget_year_ratio'] = df['budget'] / (df['release_date_year'] * df['release_date_year'])
    df['releaseYear_popularity_ratio'] = df['release_date_year'] / df['popularity']
    df['releaseYear_popularity_ratio_pd'] = df['popularity'] / df['release_date_year']
    
    df['runtime_to_mean_year'] = df['runtime'] / df.groupby("release_date_year")["runtime"].transform('mean')
    df['popularity_to_mean_year'] = df['popularity'] / df.groupby("release_date_year")["popularity"].transform('mean')
    df['budget_to_mean_year'] = df['budget'] / df.groupby("release_date_year")["budget"].transform('mean')
        
    return df

In [ ]:
new_train = new_features(train)
new_test = new_features(test)

## Data preprocessing

### Check missing values in new features

In [ ]:
new_train_inf_val = new_train[new_train==np.inf].sum().sort_values(ascending=False)
new_train_missing_val = new_train.isna().sum().sort_values(ascending=False)


inf_cols = new_train_inf_val[new_train_inf_val!=0].index.tolist()
nan_cols = new_train_missing_val[new_train_missing_val!=0].index.tolist()

display(new_train_inf_val[new_train_inf_val!=0])
display(new_train_missing_val[new_train_missing_val!=0])

In [ ]:
for col in inf_cols:
    mean_val = new_train[col].replace([np.inf, -np.inf], np.nan).mean()
    new_train[col] = new_train[col].replace([np.inf, -np.inf], mean_val)

for col in nan_cols:
    mean_val = new_train[col].replace([np.inf, -np.inf], np.nan).mean()
    new_train[col] = new_train[col].fillna(mean_val)

In [ ]:
new_test_inf_val = new_test[new_test==np.inf].sum().sort_values(ascending=False)
new_test_missing_val = new_test.isna().sum().sort_values(ascending=False)


inf_cols = new_test_inf_val[new_test_inf_val!=0].index.tolist()
nan_cols = new_test_missing_val[new_test_missing_val!=0].index.tolist()

display(new_test_inf_val[new_test_inf_val!=0])
display(new_test_missing_val[new_test_missing_val!=0])

In [ ]:
for col in inf_cols:
    mean_val = new_test[col].replace([np.inf, -np.inf], np.nan).mean()
    new_test[col] = new_test[col].replace([np.inf, -np.inf], mean_val)

for col in nan_cols:
    mean_val = new_test[col].replace([np.inf, -np.inf], np.nan).mean()
    new_test[col] = new_test[col].fillna(mean_val)

## Modeling 

In [ ]:
new_train.select_dtypes(include='object').columns

In [ ]:
new_train = new_train.reset_index(drop=True)
new_test = new_test.reset_index(drop=True)

new_train = new_train.rename(columns={
                                    'unknown/non-binary_crew': 'unknown_non_binary_crew',
                                    'unknown/non-binary': 'unknown_non-binary',
                                    'releaseYear_popularity_ratio2': 'releaseYear_popularity_ratio_pd'
                                     }
                            )
new_test = new_test.rename(columns={
                                    'unknown/non-binary_crew': 'unknown_non_binary_crew',
                                    'unknown/non-binary': 'unknown_non-binary',
                                    'releaseYear_popularity_ratio2': 'releaseYear_popularity_ratio_pd'
                                     }
                            )


new_train.columns = new_train.columns.str.replace(' ', '_')
new_test.columns = new_test.columns.str.replace(' ', '_')


X_train = new_train.drop(columns=['id', 'revenue', 'log_revenue'])
y_train_log = np.log1p(new_train['revenue'])
y_train = new_train['revenue']

X_test = new_test.drop(columns=['id', 'revenue', 'log_revenue'])
y_test_log = np.log1p(new_test['revenue'])
y_test = new_test['revenue']


X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

Напишем функцию для обучения модели, которая на вход принимает название модели, её параметры, а также обучающую и тестовую выборки, а на выход возвращает out-of-fold предсказания, предсказания на тестовой выборке, усреднённую метрику и график с наиболее значимыми признаками. 

In [ ]:
n_fold = 5
random_seed = 42
folds = KFold(n_splits=n_fold, shuffle=True, random_state=42)

def train_model(X, y, X_test, params=None, model_type='cat', plot_feature_importance=True):
    cat_cols = X.select_dtypes(include=['category','object']).columns.tolist()
    oof =  np.zeros(X.shape[0]) # out-of-fold предсказания
    prediction = np.zeros(X_test.shape[0])
    scores_rmse = []
    scores_mae = []
    scores_mape = []
    scores_r2 = []
    feature_importance = pd.DataFrame()
    for fold_n, (x_train_index, x_valid_index) in enumerate(folds.split(X)):
        print('Fold', fold_n, 'started at', time.ctime())
        y_train = y.iloc[x_train_index]
        y_valid = y.iloc[x_valid_index]
        if model_type == 'lgb':                                                  # LGBMRegressor
            X.columns = X.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
            X_test.columns = X_test.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
            X[cat_cols] = X[cat_cols].astype('category')
            X_test[cat_cols] = X_test[cat_cols].astype('category')
            X_train = X.iloc[x_train_index]
            X_valid = X.iloc[x_valid_index]
            model = LGBMRegressor(**params, 
                                  n_estimators=2000, 
                                  objective='regression',
                                  metric='rmse', 
                                  categorical_feature=cat_cols, 
                                  n_jobs=-1, 
                                  verbose=-1)
        if model_type == 'cat':                                               # CatBoostRegressor
            X_train = X.iloc[x_train_index]
            X_valid = X.iloc[x_valid_index]
            model = CatBoostRegressor(**params, 
                                      loss_function='RMSE',
                                      iterations=2000, 
                                      cat_features = cat_cols, 
                                      thread_count=-1, 
                                      verbose=0)
        if model_type == 'xgb':                                             # XGBRegressor
            for col in cat_cols:
                X[col] = X[col].astype('category').cat.codes
                X_test[col] = X_test[col].astype('category').cat.codes
            X_train = X.iloc[x_train_index]
            X_valid = X.iloc[x_valid_index]
            model = XGBRegressor(**params, 
                                 objective='reg:squarederror',
                                 n_estimators=2000, 
                                 enable_categorical=True, 
                                 n_jobs=-1, 
                                 verbose=0)
        if model_type == 'rf':                                      # RandomForestRegressor
            X_train = X.iloc[x_train_index]
            X_valid = X.iloc[x_valid_index]
            model = RandomForestRegressor(**params, 
                                          criterion='squared_error',
                                          n_estimators=500, 
                                          n_jobs=-1, 
                                          verbose=0)

        model.fit(X_train, y_train)
        y_pred_valid = model.predict(X_valid)
        y_pred = model.predict(X_test)

        oof[x_valid_index] =  y_pred_valid.reshape(-1,)
        prediction += y_pred                            # предсказания на каждом фолде суммируются
        scores_rmse.append(rmse(y_valid, y_pred_valid))
        scores_mae.append(mae(y_valid, y_pred_valid))
        scores_mape.append(mape(y_valid, y_pred_valid))
        scores_r2.append(r2_score(y_valid, y_pred_valid))

        if plot_feature_importance:
            fold_importance = pd.DataFrame()
            fold_importance['feature'] = X.columns
            fold_importance['importance'] = model.feature_importances_
            fold_importance['fold'] = fold_n+1
            feature_importance = pd.concat([feature_importance, fold_importance], axis=0)
    
    prediction /= n_fold                              # Разделив на число фолдов получим среднее предсказание по фолдам
    scores_df = pd.DataFrame({
    'LOG_RMSE': [round(np.mean(scores_rmse), 3)],
    'LOG_MAE': [round(np.mean(scores_mae), 3)],
    'LOG_MAPE': [round(np.mean(scores_mape), 3)],
    'LOG_R2_SCORE': [round(np.mean(scores_r2), 3)]
    })
    display(scores_df)


    
    if plot_feature_importance:
        cols = feature_importance[["feature", "importance"]].groupby("feature").mean().sort_values(
        by="importance", ascending=False)[:50].index    # Т.к. каждая фича оценивается количество фолдов раз, получаем среднюю важность по фолдам  
        best_features = feature_importance.loc[feature_importance.feature.isin(cols)]
        plt.figure(figsize=(16, 30));
        sns.barplot(x="importance", y="feature", data=best_features.sort_values(by="importance", ascending=False), palette=palette);
        plt.title(f'{model_type} Features (avg over folds)', fontdict=fontdict_title)  
        plt.tick_params(axis='y', labelsize=18, labelcolor='#682F2F', width=1.5)



        plt.show()

        return oof, prediction, scores_df, feature_importance, model

    return oof, prediction, scores_df, model

## Model LGBMRegressor

In [ ]:
X_train.select_dtypes(include=['category', 'object'])

In [ ]:
%%time
params = {'num_leaves': 30,
         'min_data_in_leaf': 10,
         'max_depth': 5,
         'learning_rate': 0.01,
         "boosting_type": "gbdt",
         "reg_alpha ": 0.2}
oof_lgb, prediction_lgb, log_scores_df_lgb, feature_importance_lgb, model_lgb = train_model(X_train, y_train_log, X_test, params=params, model_type='lgb')

### Metrics

In [ ]:
def get_scores_df(y_test, prediction, model_name:str, cur_scores_df:pd.DataFrame=None):
    """
    Рассчитывает и возвращает метрики качества модели (RMSE, MAE, MAPE, R²).

    Args:
        y_test (pd.Series or np.ndarray): Реальные значения целевой переменной.
        prediction (np.ndarray): Логарифмированные предсказания модели.
        model_name (str): Название модели, под которым сохраняются метрики.
        cur_scores_df (pd.DataFrame, optional): 
            Текущая таблица с метриками других моделей (если нужно добавить результаты).
            По умолчанию None.

    Returns:
        pd.DataFrame: Таблица с рассчитанными метриками (RMSE, MAE, MAPE, R²).

    Notes:
        Функция предполагает, что `prediction` даны в логарифмированной шкале
        (после np.log1p), и внутри выполняется обратное преобразование (np.expm1).
    """
    predict = np.expm1(prediction)
    rmse_score = rmse(y_test, predict)
    mae_score = mae(y_test, predict)
    mape_score = mape(y_test, predict)
    r2_sc = r2_score(y_test, predict)
    scores_df = pd.DataFrame({
        'RMSE': [round(rmse_score, 3)],
        'MAE': [round(mae_score, 3)],
        'MAPE': [round(mape_score, 3)],
        'R2_SCORE': [round(r2_sc, 3)]            
    },index = [model_name])
    if  cur_scores_df is not None:
        cur_scores_df = pd.concat([cur_scores_df, scores_df], axis=0)
        pd.set_option('display.float_format', '{:,.3f}'.format)
        display (cur_scores_df)
        return cur_scores_df
    pd.set_option('display.float_format', '{:,.3f}'.format)
    display (scores_df)
    return scores_df

In [ ]:
def get_log_scores_df(new_log_scores_df, model_name:str, log_scores_df=None):
    """
    Добавляет строку с лог-метриками в общую таблицу результатов моделей.
    Args:
        new_log_scores_df (pd.DataFrame): 
            Таблица-строка с новыми лог-метриками модели.
        log_scores_df (pd.DataFrame): 
            Текущая таблица с лог-метриками других моделей 
            (если нужно добавить результаты). Может быть None.
        model_name (str): 
            Имя модели, которое используется как индекс новой строки.
    Returns:
        pd.DataFrame: 
            Обновлённая таблица метрик с добавленными результатами модели 
            (LOG_RMSE, LOG_MAE, LOG_MAPE, LOG_R2_SCORE).
    """
    new_log_scores_df.index=[model_name] 
    if log_scores_df is not None:
        df = pd.concat([log_scores_df, new_log_scores_df], axis=0)
    else:
        df = new_log_scores_df
    display (df)
    return df

In [ ]:
cur_scores_df = get_scores_df(y_test, prediction_lgb, "model_lgb")

In [ ]:
log_scores_df = get_log_scores_df(log_scores_df_lgb, 'model_lgb') 

In [ ]:
model_lgb

## Model CatBoostRegressor

In [ ]:
%%time
params = {
    'min_data_in_leaf': 10,
    'max_depth': 5,
    'learning_rate': 0.01
         }
oof_cat, prediction_cat, log_scores_df_cat, feature_importance_cat, model_cat = train_model(X_train, y_train_log, X_test, params=params, model_type='cat')

### Metrics

In [ ]:
cur_scores_df = get_scores_df(y_test, prediction_cat, "model_cat", cur_scores_df)

In [ ]:
log_scores_df = get_log_scores_df(log_scores_df_cat, 'model_cat', log_scores_df)

## Model XGBRegressor 

In [ ]:
%%time
params = {
    'max_depth': 5,
    'learning_rate': 0.01,
    'max_leaves': 10
}
oof_xgb, prediction_xgb, log_scores_df_xgb, feature_importance_xgb, model_xgb = train_model(X_train, y_train_log, X_test, params=params, model_type='xgb')

### Metrics

In [ ]:
log_scores_df = get_log_scores_df(log_scores_df_xgb, 'model_xgb', log_scores_df)

In [ ]:
cur_scores_df = get_scores_df(y_test, prediction_xgb, 'model_xgb', cur_scores_df)

## Model RandomForestRegressor

In [ ]:

%%time
params = {
    'max_depth' : 5
}
oof_rf, prediction_rf, log_scores_df_rf, feature_importance_rf, model_rf = train_model(X_train, y_train_log, X_test, params=params, model_type='rf')

### Metrics

In [ ]:
log_scores_df = get_log_scores_df(log_scores_df_rf, 'model_rf', log_scores_df)

In [ ]:
cur_scores_df = get_scores_df(y_test, prediction_rf, 'model_rf', cur_scores_df)

По результатам сравнения всех моделей с базовыми параметрами ,по большому скопу метрик наиболее лучшее метрики показали модели LGBMRegressor и CatBoostRegressor. Попробую оптимизировать эти модели.

## Optimization Hyperparameters

### LGBMRegressor optimization

In [ ]:
def optuna_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 80, step=2),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, step=0.01),
        'n_estimators': trial.suggest_int('n_estimators', 300, 4000, step=200),
        'max_depth': trial.suggest_int('max_depth', 4, 16, step=2),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50, step=5),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0, step=0.1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0, step=0.1),
        'n_jobs': -1,
        'verbose': -1  # отключаем все предупреждения LightGBM
    }

    cat_cols = X_train.select_dtypes(include=['category','object']).columns.tolist()
    X = X_train.copy()
    y = y_train_log.copy()
    X[cat_cols] = X[cat_cols].astype('category')

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model = LGBMRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            categorical_feature=cat_cols
        )

        preds = model.predict(X_val)
        rmse_score = rmse(y_val, preds)
        scores.append(rmse_score)

    return np.mean(scores)


In [ ]:
%%time
study_lgb = optuna.create_study(direction='minimize', study_name='lgbm_cv')
study_lgb.optimize(optuna_lgb, n_trials=50, n_jobs=1)

In [ ]:
lgb_best_value = study_lgb.best_value
print("Оптимизированный RMSE:", round(lgb_best_value, 3))
lgb_best_params = study_lgb.best_params
print("Наиболее лучшие параметры:", lgb_best_params)

print("Возможность визуализации:", optuna.visualization.is_available())
display(optuna.visualization.plot_optimization_history(study_lgb))
display(optuna.visualization.plot_contour(study_lgb, params=['n_estimators', 'max_depth']))
display(optuna.visualization.plot_param_importances(study_lgb))

### CatBoostRegressor optimization

In [ ]:
def optuna_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 2000, step=100),
        "depth": trial.suggest_int("depth", 3, 10, step=1),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, step=0.01),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, step=0.5),
        "loss_function": "RMSE",
        "random_seed": 42,
        "thread_count": 2,
        "verbose": 0
    }

    cat_cols = X_train.select_dtypes(include=['category','object']).columns.tolist()

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train_log.iloc[train_idx], y_train_log.iloc[valid_idx]
    
        model_cat = CatBoostRegressor(
            **params,
            cat_features=cat_cols
        )
        model_cat.fit(X_tr, y_tr, verbose=0)
    
        preds = model_cat.predict(X_val)
        rmse_score = rmse(y_val, preds)
        scores.append(rmse_score)

    return np.mean(scores)    

In [ ]:
%%time
study = optuna.create_study(direction="minimize", study_name="catboost_cv")
study.optimize(optuna_cat, n_trials=50, n_jobs=1)

In [ ]:
cat_best_value = study.best_value
print("Оптимизированный RMSE:", round(cat_best_value, 3))
cat_best_params = study.best_params
print("Наиболее лучшие параметры:", cat_best_params)

print("Возможность визуализации:", optuna.visualization.is_available())
display(optuna.visualization.plot_optimization_history(study))
display(optuna.visualization.plot_contour(study, params=['iterations', 'depth']))
display(optuna.visualization.plot_param_importances(study))

## Model CatBoostRegressor and LGBMRegressor with optimized hyperparameters 

In [ ]:
n_fold = 5
random_seed = 42
folds = KFold(n_splits=n_fold, shuffle=True, random_state=42)

def train_model_opt(X, y, X_test, params=None, model_type='cat', plot_feature_importance=True):
    cat_cols = X.select_dtypes(include=['category','object']).columns.tolist()
    oof =  np.zeros(X.shape[0]) # out-of-fold предсказания
    prediction = np.zeros(X_test.shape[0])
    scores_rmse = []
    scores_mae = []
    scores_mape = []
    scores_r2 = []
    feature_importance = pd.DataFrame()
    for fold_n, (x_train_index, x_valid_index) in enumerate(folds.split(X)):
        print('Fold', fold_n, 'started at', time.ctime())
        y_train = y.iloc[x_train_index]
        y_valid = y.iloc[x_valid_index]
        if model_type == 'lgb':                                                  # LGBMRegressor
            X.columns = X.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
            X_test.columns = X_test.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
            X[cat_cols] = X[cat_cols].astype('category')
            X_test[cat_cols] = X_test[cat_cols].astype('category')
            X_train = X.iloc[x_train_index]
            X_valid = X.iloc[x_valid_index]
            model = LGBMRegressor(**params, 
                                  objective='regression',
                                  metric='rmse', 
                                  categorical_feature=cat_cols, 
                                  n_jobs=-1, 
                                  verbose=0)
        if model_type == 'cat':                                               # CatBoostRegressor
            X_train = X.iloc[x_train_index]
            X_valid = X.iloc[x_valid_index]
            model = CatBoostRegressor(**params, 
                                      loss_function='RMSE',
                                      cat_features = cat_cols, 
                                      thread_count=-1, 
                                      verbose=0)

        model.fit(X_train, y_train)
        y_pred_valid = model.predict(X_valid)
        y_pred = model.predict(X_test)

        oof[x_valid_index] =  y_pred_valid.reshape(-1,)
        prediction += y_pred                            # предсказания на каждом фолде суммируются
        scores_rmse.append(rmse(y_valid, y_pred_valid))
        scores_mae.append(mae(y_valid, y_pred_valid))
        scores_mape.append(mape(y_valid, y_pred_valid))
        scores_r2.append(r2_score(y_valid, y_pred_valid))

        if plot_feature_importance:
            fold_importance = pd.DataFrame()
            fold_importance['feature'] = X.columns
            fold_importance['importance'] = model.feature_importances_
            fold_importance['fold'] = fold_n+1
            feature_importance = pd.concat([feature_importance, fold_importance], axis=0)
    
    prediction /= n_fold                              # Разделив на число фолдов получим среднее предсказание по фолдам
    scores_df = pd.DataFrame({
    'LOG_RMSE': [round(np.mean(scores_rmse), 3)],
    'LOG_MAE': [round(np.mean(scores_mae), 3)],
    'LOG_MAPE': [round(np.mean(scores_mape), 3)],
    'LOG_R2_SCORE': [round(np.mean(scores_r2), 3)]
    })
    display(scores_df)


    
    if plot_feature_importance:
        cols = feature_importance[["feature", "importance"]].groupby("feature").mean().sort_values(
        by="importance", ascending=False)[:50].index    # Т.к. каждая фича оценивается количество фолдов раз, получаем среднюю важность по фолдам  
        best_features = feature_importance.loc[feature_importance.feature.isin(cols)]
        plt.figure(figsize=(16, 30));
        sns.barplot(x="importance", y="feature", data=best_features.sort_values(by="importance", ascending=False), palette=palette);
        plt.title(f'{model_type} Features (avg over folds)', fontdict=fontdict_title)  
        plt.tick_params(axis='y', labelsize=18, labelcolor='#682F2F', width=1.5)



        plt.show()

        return oof, prediction, scores_df, feature_importance, model

    return oof, prediction, scores_df, model

In [ ]:
oof_cat_opt, prediction_cat_opt, log_scores_df_cat_opt, feature_importance_cat_opt, model_cat_opt = train_model_opt(X_train, y_train_log, X_test, params=cat_best_params, model_type='cat')

### Metrics

In [ ]:
cur_scores_df = get_scores_df(y_test, prediction_cat_opt, "model_cat_opt", cur_scores_df)

In [ ]:
log_scores_df = get_log_scores_df(log_scores_df_cat_opt, 'model_cat_opt', log_scores_df)

In [ ]:
oof_lgb_opt, prediction_lgb_opt, log_scores_df_lgb_opt, feature_importance_lgb_opt, model_lgb_opt = train_model_opt(X_train, y_train_log, X_test, params=lgb_best_params, model_type='lgb')

### Metrics

In [ ]:
cur_scores_df = get_scores_df(y_test, prediction_lgb_opt, "model_lgb_opt", cur_scores_df)

In [ ]:
log_scores_df = get_log_scores_df(log_scores_df_lgb_opt, 'model_lgb_opt', log_scores_df)

**Итоговые выводы:**

Оптимизация гиперпараметров моделей LGBMRegressor и CatBoostRegressor позволила улучшить показатели на логарифмированных целевых данных, однако метрики, рассчитанные на исходных значениях целевой переменной, изменились незначительно. Исключением стала метрика MAPE, значение которой до оптимизации было ниже. Это может свидетельствовать о том, что в процессе подбора гиперпараметров следовало оптимизировать метрику, вычисляемую на реальных (не логарифмированных) значениях целевой переменной.

Анализ feature importance показал, что наибольший вклад в качество модели вносят признаки, отражающие характеристики фильмов: взвешенный рейтинг IMDb, состав жанров, длина описания, продолжительность фильма, а также год, дата и день недели релиза. Кроме того, высокую значимость продемонстрировали признаки, полученные на этапе feature engineering, в частности — созданные с помощью арифметических преобразований. В верхней части рейтинга по значимости оказались переменные, связанные с бюджетом фильма: его доля относительно среднего годового бюджета, абсолютное значение бюджета и производные показатели, требующие дополнительной интерпретации. Существенное влияние также оказали признаки, отражающие длительность фильма — её доля от средней длительности за год, сама длительность и отношение бюджета к длительности.

Качество модели в дальнейшем буду совершенствовать, можно попробовать, например, улучшить качество данных сопоставив кассовые сборы с источниками IMDB, где информация более точная и актуальная, попробовать оптимизировать метрики на реальных значениях выручки.

<center>
  <div style="display: flex; justify-content: center; align-items: center; gap: 20px;">
    <img src="https://i.pinimg.com/564x/8f/9f/09/8f9f099cba40f01b9bd75d788bdba741.jpg"
         width="1000" 
         height="400">
  </div>
</center>